<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9_multiclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 9 (multiclass) — a minimal CE correction of the dual hNPE–hNDE model

This notebook is an alternative to `Exercise_9_Hybrid_NPE_NDE.ipynb`; the original exercise remains unchanged.  The construction is deliberately kept close to that exercise:

1. train conditional normalizing-flow proposals by maximum likelihood;
2. freeze the proposals and build balanced simulator/posterior-proposal/likelihood-proposal classes;
3. train a plain multiclass MLP with **equal-prior cross entropy only**;
4. infer posterior and likelihood residuals from the softmax probabilities;
5. treat conditional normalization and the Bayes bridge only as post-training corrections and checks.

Part I omits the nuisance parameter and therefore learns the marginalized one-dimensional posterior proposal $q_P(\mu\mid x)$.  Part II follows the original dual construction directly: it learns one joint posterior flow $q_P(\mu,\alpha\mid x)$ and one likelihood flow $q_L(x\mid\mu,\alpha)$, both with the original Exercise-9 spline architecture and training configuration.

The correction is an ensemble of ten independently initialized classifiers. Each member contains only repeated `Linear` + `SiLU` layers and a final three-logit layer. Members train for 250 epochs with batch size 1024 and a progressive learning rate from $10^{-4}$ to $10^{-9}$, divided by ten every 40 epochs. There is no dropout, weight decay, layer normalization, residual block, output clipping, auxiliary binary objective, normalization loss, or bridge loss. Inference uses the arithmetic mean of the ten positive softmax-ratio estimates. Analytic Gaussian densities are used only for validation plots.


In [ ]:
# ========================================================================
# Google Colab setup — safe to re-run; a no-op outside Colab.
# ========================================================================
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
USE_DRIVE = True

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab")
    else:
        ROOT = Path("/content")
    ROOT.mkdir(parents=True, exist_ok=True)

    REPO_DIR = ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    WORK_DIR = REPO_DIR / "workshops" / "ml4hep_tifr"
    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )
    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        import_path = str(import_dir.resolve())
        if import_path not in sys.path:
            sys.path.insert(0, import_path)
    run(sys.executable, "-m", "pip", "install", "-q", "nflows==0.14")
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)
else:
    candidates = [Path.cwd(), Path.cwd() / "workshops" / "ml4hep_tifr_colab"]
    TUTORIAL_DIR = next(
        (candidate for candidate in candidates if (candidate / "utils_hnpe.py").exists()),
        None,
    )
    if TUTORIAL_DIR is None:
        raise FileNotFoundError(
            "Run from the repository root or workshops/ml4hep_tifr_colab."
        )
    if str(TUTORIAL_DIR.resolve()) not in sys.path:
        sys.path.insert(0, str(TUTORIAL_DIR.resolve()))


## What is trained, and how ratios are read out

In either part the three equal-prior class densities are denoted by $(\Pi_S,\Pi_P,\Pi_L)$.  A classifier trained with the single objective

$$
\mathcal L_{\rm CE}=-\mathbb E\log d_y(\vartheta,x)
$$

returns the softmax probabilities

$$
(d_S,d_P,d_L)=\operatorname{softmax}(s_S,s_P,s_L).
$$

Equal class priors then give the two residual density ratios directly:

$$
r_P=\frac{\Pi_S}{\Pi_P}=\frac{d_S}{d_P},\qquad
r_L=\frac{\Pi_S}{\Pi_L}=\frac{d_S}{d_L}.
$$

Each ensemble member evaluates softmax in float64 and forms these probability quotients. The implementation averages the ten positive quotient estimates arithmetically and never explicitly evaluates `exp(logit_difference)`. When a log density is required, it takes the logarithm of that arithmetic ensemble ratio. Softmax itself necessarily contains a stabilized exponential internally; the gain is that the ratio interface is a normalized probability output rather than an explicitly exponentiated unbounded logit.

Conditional masses such as $Z_P=\mathbb E_{q_P}[r_P]$ and $Z_L=\mathbb E_{q_L}[r_L]$ are estimated only after the classifier is frozen.  The bridge is also only a consistency check.  Neither quantity enters training, validation selection, or any gradient.


In [ ]:
import copy
import gc
import hashlib
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from scipy.special import logsumexp
from scipy.stats import multivariate_normal, norm, wasserstein_distance
from torch.utils.data import DataLoader, TensorDataset

from utils_dual_hnde import importance_tail_summary
from utils_hnpe import (
    sample_spline_flow,
    scalar_spline_flow_cdf,
    scalar_spline_flow_icdf,
    spline_flow_log_prob,
    train_spline_flow,
)
from utils_plotting import export_standalone_figure_script


SEED = 19092026
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


def set_torch_seed(seed):
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def _flow_log_prob(flow_pack, target, *, context=None, batch_size=65_536):
    return spline_flow_log_prob(
        flow_pack, target, context=context, batch_size=batch_size
    )


SMOKE_MODE = False
FAST_MODE = False
LOAD_IF_AVAILABLE = False

if SMOKE_MODE:
    RUN_TAG = "smoke"
    N_FLOW, N_CLASS = 2_000, 3_000
    FLOW_EPOCHS, CLASS_EPOCHS, CLASSIFIER_ENSEMBLE_SIZE = 2, 4, 1
    N_DIAGNOSTIC_REFERENCE = 64
    N_FLOW_AUDIT_SAMPLES = 128
    N_BRIDGE_AUDIT_GROUPS, N_BRIDGE_AUDIT_INNER = 12, 8
    N_BRIDGE_AUDIT_MASS_INNER, N_BRIDGE_AUDIT_BANKS = 8, 1
elif FAST_MODE:
    RUN_TAG = "fast"
    N_FLOW, N_CLASS = 35_000, 80_000
    FLOW_EPOCHS, CLASS_EPOCHS, CLASSIFIER_ENSEMBLE_SIZE = 12, 250, 10
    N_DIAGNOSTIC_REFERENCE = 256
    N_FLOW_AUDIT_SAMPLES = 1_024
    N_BRIDGE_AUDIT_GROUPS, N_BRIDGE_AUDIT_INNER = 96, 24
    N_BRIDGE_AUDIT_MASS_INNER, N_BRIDGE_AUDIT_BANKS = 16, 2
else:
    RUN_TAG = "full"
    # The original Exercise 9 uses 250k flow rows and 500k ratio pairs.
    N_FLOW, N_CLASS = 250_000, 500_000
    FLOW_EPOCHS, CLASS_EPOCHS, CLASSIFIER_ENSEMBLE_SIZE = 50, 250, 10
    N_DIAGNOSTIC_REFERENCE = 768
    N_FLOW_AUDIT_SAMPLES = 16_384
    N_BRIDGE_AUDIT_GROUPS, N_BRIDGE_AUDIT_INNER = 512, 48
    N_BRIDGE_AUDIT_MASS_INNER, N_BRIDGE_AUDIT_BANKS = 32, 4

N_GRID_REFERENCE = 48 if SMOKE_MODE else (128 if FAST_MODE else 256)
N_CALIBRATION_CONTEXTS = 20 if SMOKE_MODE else (200 if FAST_MODE else 2_000)
N_CALIBRATION_SAMPLES = 128 if SMOKE_MODE else (1_024 if FAST_MODE else 2_048)
N_JOINT_CALIBRATION_CONTEXTS = 12 if SMOKE_MODE else (100 if FAST_MODE else 500)
N_JOINT_CALIBRATION_SAMPLES = 128 if SMOKE_MODE else (512 if FAST_MODE else 1_024)

MODEL_DIR = Path("models_exercise9_multiclass_v9_classifier_ensemble") / RUN_TAG
FIGURE_SCRIPT_DIR = Path("exercise9_multiclass_v9_figures_scripts") / RUN_TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_SCRIPT_DIR.mkdir(parents=True, exist_ok=True)

# Exact model and optimizer configuration from the original dual Exercise 9.
FLOW_MODEL_CONFIG = {
    "n_coupling_layers": 10,
    "hidden_features": 512,
    "hidden_layers": 4,
    "spline_num_bins": 16,
    "spline_tail_bound": 5.0,
    "dropout_probability": 0.0,
}
FLOW_TRAINING_CONFIG = {
    "batch_size": 1024,
    "n_epochs": FLOW_EPOCHS,
    "learning_rate": 1.0e-3,
    "min_learning_rate": 1.0e-11,
    "validation_fraction": 0.2,
    "patience": 10,
    "gradient_clip": 5.0,
}

# Part I has a scalar posterior target, for which a single conditional
# monotone spline is the appropriate one-dimensional analogue.
SCALAR_FLOW_MODEL_CONFIG = {
    "n_coupling_layers": 1,
    "hidden_features": 512,
    "hidden_layers": 5,
    "spline_num_bins": 32,
    "spline_tail_bound": 12.0,
    "dropout_probability": 0.0,
    "identity_initialization": True,
}
SCALAR_FLOW_TRAINING_CONFIG = {
    **FLOW_TRAINING_CONFIG,
    "learning_rate": 3.0e-4,
    "min_learning_rate": 1.0e-9,
    "gradient_clip": 1.0,
    "lr_scheduler_patience": 2,
    "retain_initial_model": True,
}

# Every ensemble member is the same plain, uniform MLP.
# There is no regularization or auxiliary loss.
CLASS_MODEL_CONFIG = {
    "hidden_features": 1024 if not SMOKE_MODE else 128,
    "hidden_layers": 4 if not SMOKE_MODE else 2,
}
CLASS_TRAINING_CONFIG = {
    "batch_size": 1024 if not SMOKE_MODE else 256,
    "validation_batch_size": 8192 if not SMOKE_MODE else 512,
    "n_epochs": CLASS_EPOCHS,
    "learning_rate": 1.0e-4,
    "min_learning_rate": 1.0e-9,
    "lr_scheduler_factor": 0.1,
    "lr_scheduler_step_epochs": 40,
    "validation_fraction": 0.2,
    "patience": CLASS_EPOCHS,
    "minimum_improvement": 1.0e-6,
}

THREE_CE_LABEL = "multiclass CE"
JOINT_CE_LABEL = "joint multiclass CE"
SIMULATOR_SIGMA = np.array([0.95, 0.38, 0.30], dtype=float)
X_OBS = np.array([0.40, 1.35, 0.12], dtype=float)


def export_exercise9_multiclass_figure(fig, script_name):
    path = export_standalone_figure_script(
        fig, script_name=script_name, output_dir=FIGURE_SCRIPT_DIR
    )
    png_path = FIGURE_SCRIPT_DIR / f"{Path(script_name).stem}.png"
    pdf_path = FIGURE_SCRIPT_DIR / f"{Path(script_name).stem}.pdf"
    fig.savefig(png_path, dpi=220, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print("Exported:", path, png_path, pdf_path)
    return path


print(
    f"mode={RUN_TAG}, N_flow={N_FLOW:,}, N_class={N_CLASS:,}, "
    f"classifier_ensemble={CLASSIFIER_ENSEMBLE_SIZE}"
)
print("Hybrid Part II flows: original Exercise-9 joint q_P(theta|x) and q_L(x|theta)")
print("Ratio model: plain MLP, equal-prior multiclass CE only")


## The unchanged Gaussian simulator and exact validation densities

We retain Exercise 9 exactly:

$$
\begin{aligned}
  x_1&\sim\mathcal N(\mu+0.8\alpha,0.95^2),\\
  x_2&\sim\mathcal N(0.72\mu^2-0.4\alpha,0.38^2),\\
  x_3&\sim\mathcal N(0.8\cos\mu+0.3\alpha,0.30^2),
\end{aligned}
$$

with $\rho_\mu=0.9\mathcal N(0,1.5^2)+0.1\mathcal N(0,4^2)$ and
$\rho_\alpha=0.9\mathcal N(0,1^2)+0.1\mathcal N(0,3^2)$.

When $\alpha$ is hidden, it can be integrated analytically for validation.  Writing
$b(\mu)=(\mu,0.72\mu^2,0.8\cos\mu)$ and $v=(0.8,-0.4,0.3)$,

$$
  p_m(x\mid\mu)=0.9\,\mathcal N(x;b,\Sigma_\epsilon+vv^T)
  +0.1\,\mathcal N(x;b,\Sigma_\epsilon+9vv^T).
$$

Neither this expression nor the explicit analytic likelihood is passed to a network.


In [ ]:
def _mixture_logpdf(values, core_sigma, broad_sigma):
    values = np.asarray(values, dtype=float)
    return logsumexp(
        np.stack([
            np.log(0.9) + norm.logpdf(values, 0.0, core_sigma),
            np.log(0.1) + norm.logpdf(values, 0.0, broad_sigma),
        ]),
        axis=0,
    )

def design_mu_logpdf(mu):
    return _mixture_logpdf(mu, 1.5, 4.0)

def design_alpha_logpdf(alpha):
    return _mixture_logpdf(alpha, 1.0, 3.0)

def design_logpdf(theta):
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    return design_mu_logpdf(theta[:, 0]) + design_alpha_logpdf(theta[:, 1])

def _sample_mixture(n, core_sigma, broad_sigma, rng):
    broad = rng.random(int(n)) < 0.1
    sigma = np.where(broad, broad_sigma, core_sigma)
    return rng.normal(0.0, sigma)

def sample_mu(n, rng):
    return _sample_mixture(n, 1.5, 4.0, rng).astype(np.float32)

def sample_alpha(n, rng):
    return _sample_mixture(n, 1.0, 3.0, rng).astype(np.float32)

def sample_design(n, rng):
    return np.column_stack([sample_mu(n, rng), sample_alpha(n, rng)]).astype(
        np.float32
    )

def simulator_base_mean(mu):
    mu = np.asarray(mu, dtype=float).ravel()
    return np.column_stack([mu, 0.72 * mu**2, 0.8 * np.cos(mu)])

def simulator_mean(theta):
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    mu, alpha = theta[:, 0], theta[:, 1]
    return simulator_base_mean(mu) + alpha[:, None] * np.array([0.8, -0.4, 0.3])

def simulate(theta, rng):
    mean = simulator_mean(theta)
    return (mean + rng.normal(size=mean.shape) * SIMULATOR_SIGMA).astype(
        np.float32
    )

def log_likelihood(x, theta):
    """Analytic truth used only in validation cells."""
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    x = np.atleast_2d(np.asarray(x, dtype=float))
    if len(x) == 1 and len(theta) > 1:
        x = np.repeat(x, len(theta), axis=0)
    if len(x) != len(theta):
        raise ValueError("x and theta must have matching rows or one x row.")
    return np.sum(norm.logpdf(x, simulator_mean(theta), SIMULATOR_SIGMA), axis=1)

_NOISE_COV = np.diag(SIMULATOR_SIGMA**2)
_ALPHA_LOADING = np.array([0.8, -0.4, 0.3])
_MARGINAL_COVS = [
    _NOISE_COV + sigma_alpha**2 * np.outer(_ALPHA_LOADING, _ALPHA_LOADING)
    for sigma_alpha in (1.0, 3.0)
]

def marginal_log_likelihood(x, mu):
    """Exact p_m(x|mu) after integrating the design nuisance mixture."""
    mu = np.asarray(mu, dtype=float).ravel()
    x = np.atleast_2d(np.asarray(x, dtype=float))
    if len(x) == 1 and len(mu) > 1:
        x = np.repeat(x, len(mu), axis=0)
    if len(x) != len(mu):
        raise ValueError("x and mu must have matching rows or one x row.")
    residual = x - simulator_base_mean(mu)
    components = [
        np.log(weight)
        + np.atleast_1d(
            multivariate_normal.logpdf(residual, mean=np.zeros(3), cov=cov)
        )
        for weight, cov in zip((0.9, 0.1), _MARGINAL_COVS)
    ]
    return np.atleast_1d(logsumexp(np.stack(components), axis=0))

def normalize_log_curve(log_density, grid):
    log_density = np.asarray(log_density, dtype=float)
    shift = float(np.max(log_density))
    density = np.exp(log_density - shift)
    integral = np.trapezoid(density, grid)
    return density / integral, shift + np.log(integral)

def normalize_log_surface(log_density, x_grid, y_grid):
    log_density = np.asarray(log_density, dtype=float)
    shift = float(np.max(log_density))
    density = np.exp(log_density - shift)
    integral = np.trapezoid(np.trapezoid(density, y_grid, axis=1), x_grid)
    return density / integral, shift + np.log(integral)

def integrated_absolute_error(reference, estimate, grid):
    return float(np.trapezoid(np.abs(reference - estimate), grid))

def js_distance_discrete(reference, estimate, floor=1.0e-15):
    reference = np.asarray(reference, dtype=float).ravel() + floor
    estimate = np.asarray(estimate, dtype=float).ravel() + floor
    reference /= reference.sum()
    estimate /= estimate.sum()
    middle = 0.5 * (reference + estimate)
    divergence = 0.5 * np.sum(reference * np.log(reference / middle))
    divergence += 0.5 * np.sum(estimate * np.log(estimate / middle))
    return float(np.sqrt(max(0.0, divergence)))

print("Observed x:", X_OBS)
print("Truth check, log p_m(x_obs|mu=1.2):", marginal_log_likelihood(X_OBS, [1.2])[0])



def sample_tail_enriched_design(n_samples, rng, tail_fraction=0.25):
    """Sample held-out bridge contexts with extra empirical tail coverage.

    These contexts are constructed only after training to stress-test the
    normalization and bridge relations across the design support.  They do
    not affect a loss, validation score, or checkpoint.  Selection uses
    sampled parameter ranks; no simulator density is evaluated.
    """

    n_samples = int(n_samples)
    n_tail = min(n_samples, max(1, int(round(tail_fraction * n_samples))))
    bulk = sample_design(n_samples - n_tail, rng)
    pool = sample_design(max(1_024, 12 * n_tail), rng)
    center = np.median(pool, axis=0)
    mad = 1.4826 * np.median(np.abs(pool - center), axis=0)
    scale = np.where(mad > 1.0e-6, mad, pool.std(axis=0))
    scale = np.where(scale > 1.0e-6, scale, 1.0)
    score = np.max(np.abs((pool - center) / scale), axis=1)
    tail_pool = np.flatnonzero(score >= np.quantile(score, 0.90))
    selected = rng.choice(tail_pool, size=n_tail, replace=len(tail_pool) < n_tail)
    combined = np.concatenate([bulk, pool[selected]], axis=0)
    return combined[rng.permutation(len(combined))].astype(np.float32)


## A plain sample-only multiclass ratio estimator

The classifier receives only sampled coordinates.  Inputs are standardized with the training-sample mean and standard deviation.  Full and fast modes use four identical `Linear(1024)` + `SiLU` layers; smoke mode uses two `Linear(128)` + `SiLU` layers solely for a quick code-path test.  Both end in three unconstrained logits.  There are no residual connections, normalization layers, dropout layers, bounded outputs, or auxiliary heads.

Each simulator/proposal group is assigned wholly to training or validation before its three rows are flattened. Thus paired rows cannot leak across the split. All classes occur exactly once per group, so their empirical priors are equal. Ten independent Adam fits minimize ordinary multiclass cross entropy for 250 epochs with batch size 1024. The learning rate is $10^{-4}$ for epochs 1--40 and is divided by ten every 40 epochs, reaching $10^{-9}$ for epochs 201--250. Patience spans the full schedule, and held-out multiclass CE alone selects each member's checkpoint.

Ratios are evaluated from float64 softmax probabilities.  No density, analytic score, simulator mean, or handcrafted residual is an input to the network.


In [ ]:
class PlainMulticlassMLP(nn.Module):
    # Uniform Linear-SiLU stack followed by three unconstrained logits.

    def __init__(self, input_dim, n_classes, hidden_features, hidden_layers):
        super().__init__()
        layers = []
        width_in = int(input_dim)
        for _ in range(int(hidden_layers)):
            layers.extend([nn.Linear(width_in, int(hidden_features)), nn.SiLU()])
            width_in = int(hidden_features)
        layers.append(nn.Linear(width_in, int(n_classes)))
        self.network = nn.Sequential(*layers)
        self.input_dim = int(input_dim)
        self.n_classes = int(n_classes)

    def forward(self, values):
        return self.network(values)
def _assert_finite(name, values, ndim=None):
    values = np.asarray(values)
    if ndim is not None and values.ndim != ndim:
        raise ValueError(f"{name} must have ndim={ndim}; got {values.shape}.")
    if not np.isfinite(values).all():
        raise ValueError(f"{name} contains non-finite values.")
    return values

def _install_nflows_rqs_float64_retry():
    """Retry only a failed float32 inverse-RQS kernel in float64.

    For a monotone rational-quadratic spline the inverse discriminant is
    non-negative analytically.  nflows 0.14 evaluates it in float32, where a
    nearly double root can acquire a tiny negative value by cancellation.  We
    retry the *same* spline tensors in float64: no row is dropped, clipped, or
    resampled.  The original float64 assertion remains the hard guard.
    """
    import functools
    import importlib
    import inspect
    from importlib.metadata import version
    import warnings

    nflows_version = version("nflows")
    if nflows_version != "0.14":
        raise RuntimeError(
            "This audited numerical guard requires nflows==0.14; "
            f"found {nflows_version}."
        )
    module = importlib.import_module(
        "nflows.transforms.splines.rational_quadratic"
    )
    original = module.rational_quadratic_spline
    if getattr(original, "_exercise9_float64_retry", False):
        return original
    signature = inspect.signature(original)

    @functools.wraps(original)
    def guarded(*args, **kwargs):
        inputs_fast = kwargs.get("inputs", args[0] if args else None)
        inverse_fast = kwargs.get(
            "inverse", args[4] if len(args) > 4 else False
        )
        if (
            inverse_fast
            and torch.is_tensor(inputs_fast)
            and inputs_fast.dtype == torch.float32
        ):
            guarded._float32_inverse_call_count += 1
            guarded._float32_inverse_values += int(inputs_fast.numel())
        try:
            return original(*args, **kwargs)
        except AssertionError as error32:
            bound = signature.bind(*args, **kwargs)
            inputs = bound.arguments["inputs"]
            inverse = bound.arguments.get(
                "inverse", signature.parameters["inverse"].default
            )
            if not inverse or inputs.dtype != torch.float32:
                raise

            floating = [
                value for value in (*args, *kwargs.values())
                if torch.is_tensor(value) and value.is_floating_point()
            ]
            if any(not bool(torch.isfinite(value).all()) for value in floating):
                raise FloatingPointError(
                    "Non-finite tensor reached the inverse RQS; refusing the "
                    "precision retry."
                ) from error32

            def to_float64(value):
                if torch.is_tensor(value) and value.is_floating_point():
                    return value.to(dtype=torch.float64)
                return value

            try:
                outputs64, logdet64 = original(
                    *(to_float64(value) for value in args),
                    **{
                        name: to_float64(value)
                        for name, value in kwargs.items()
                    },
                )
            except AssertionError as error64:
                raise RuntimeError(
                    "The inverse-RQS discriminant also failed in float64. "
                    "Refusing to clip or resample; retrain this flow."
                ) from error64
            if not (
                bool(torch.isfinite(outputs64).all())
                and bool(torch.isfinite(logdet64).all())
            ):
                raise FloatingPointError(
                    "The float64 inverse-RQS retry returned non-finite values."
                ) from error32

            guarded._float64_retry_count += 1
            guarded._float64_retry_values += int(inputs.numel())
            if guarded._float64_retry_count == 1:
                warnings.warn(
                    "nflows float32 inverse-RQS cancellation: retrying the "
                    "same spline call in float64.",
                    RuntimeWarning,
                    stacklevel=2,
                )
            outputs = outputs64.to(dtype=inputs.dtype)
            logdet = logdet64.to(dtype=inputs.dtype)
            if not (
                bool(torch.isfinite(outputs).all())
                and bool(torch.isfinite(logdet).all())
            ):
                raise FloatingPointError(
                    "Casting the inverse-RQS retry back to float32 "
                    "produced non-finite values."
                ) from error32
            return outputs, logdet

    guarded._exercise9_float64_retry = True
    guarded._float64_retry_count = 0
    guarded._float64_retry_values = 0
    guarded._float32_inverse_call_count = 0
    guarded._float32_inverse_values = 0
    guarded._float32_original = original
    module.rational_quadratic_spline = guarded

    # nflows 0.14's linear-tail helper resolves the module global above.  The
    # aliases cover any direct bounded-spline call without touching package
    # source or a checkpoint.
    importlib.import_module(
        "nflows.transforms.splines"
    ).rational_quadratic_spline = guarded
    importlib.import_module(
        "nflows.transforms.autoregressive"
    ).rational_quadratic_spline = guarded
    linear_tail = importlib.import_module(
        "nflows.transforms.splines"
    ).unconstrained_rational_quadratic_spline
    if linear_tail.__globals__.get("rational_quadratic_spline") is not guarded:
        raise RuntimeError(
            "The nflows 0.14 linear-tail inverse did not bind to the "
            "audited RQS guard."
        )
    return guarded


RQS_NUMERIC_GUARD = _install_nflows_rqs_float64_retry()


def _rqs_retry_count():
    return int(getattr(RQS_NUMERIC_GUARD, "_float64_retry_count", 0))


def _rqs_inverse_call_count():
    return int(
        getattr(RQS_NUMERIC_GUARD, "_float32_inverse_call_count", 0)
    )


def _draw_conditional(flow_pack, contexts, n_samples, seed, allocation=None):
    # allocation is retained as a compatibility no-op for old call sites;
    # This notebook uses one proposal flow, hence no component labels to allocate.
    del allocation
    contexts = np.atleast_2d(np.asarray(contexts, dtype=np.float32))
    n_samples = int(n_samples)
    n_features = int(flow_pack["config"]["n_features"])
    set_torch_seed(seed)
    draws = sample_spline_flow(
        flow_pack, n_samples, context=contexts, batch_size=16_384
    )
    draws = np.asarray(draws, dtype=np.float32)
    if len(contexts) == 1:
        draws = draws[None, :, :]
    expected = (len(contexts), n_samples, n_features)
    if draws.shape != expected:
        raise RuntimeError(
            f"Unexpected conditional sample shape {draws.shape}; expected {expected}."
        )
    return _assert_finite("conditional flow draws", draws, ndim=3)



def _audit_context_subset(flow_pack, contexts, seed, n_contexts=2_048):
    contexts = _assert_finite("flow audit contexts", contexts, ndim=2).astype(
        np.float32
    )
    if len(contexts) < n_contexts:
        raise ValueError("The flow audit needs at least n_contexts rows.")
    scaler = flow_pack["context_scaler"]
    standardized = (contexts - scaler.mean) / scaler.std
    extremeness = np.max(np.abs(standardized), axis=1)
    n_tail = min(512, n_contexts // 4)
    tail_index = np.argpartition(extremeness, -n_tail)[-n_tail:]
    available = np.setdiff1d(
        np.arange(len(contexts)), tail_index, assume_unique=False
    )
    rng = np.random.default_rng(int(seed))
    typical_index = rng.choice(
        available, size=n_contexts - n_tail, replace=False
    )
    return contexts[np.concatenate([typical_index, tail_index])]


def audit_scalar_target_support(flow_pack, target_anchors, name):
    """Require explicit audit anchors to lie inside the spline box."""
    target_anchors = np.asarray(target_anchors, dtype=np.float64).reshape(-1, 1)
    scaler = flow_pack["target_scaler"]
    standardized = (
        target_anchors - np.asarray(scaler.mean, dtype=np.float64)
    ) / np.asarray(scaler.std, dtype=np.float64)
    max_abs = float(np.max(np.abs(standardized)))
    bound = float(flow_pack["config"]["spline_tail_bound"])
    if max_abs >= bound:
        raise RuntimeError(
            f"{name}: audit anchors reach {max_abs:.3f} standardized units, "
            f"outside spline bound {bound:.3f}.  This conditional RQS is "
            "context-independent in that tail; widen the scalar domain "
            "rather than accepting an inverse retry."
        )
    print(
        f"{name} scalar support: max |standardized anchor|={max_abs:.3f} "
        f"< spline bound {bound:.3f}."
    )


def audit_scalar_flow(flow_pack, contexts, name, seed):
    """Active inverse/forward, density, finiteness, and RNG checks."""
    if int(flow_pack["config"]["n_features"]) != 1:
        raise ValueError("audit_scalar_flow requires a scalar target flow.")
    history = flow_pack.get("history", {})
    initial_values = history.get("initial_validation", [])
    selected_values = history.get("selected_validation", [])
    if initial_values and selected_values:
        initial_validation = float(initial_values[-1])
        selected_validation = float(selected_values[-1])
        print(
            f"{name} validation NLL: identity={initial_validation:.4f}, "
            f"selected={selected_validation:.4f}."
        )
        if selected_validation >= initial_validation - 1.0e-4:
            import warnings
            warnings.warn(
                f"{name} retained its context-independent identity "
                "baseline; numerical closure can still pass, but proposal "
                "fidelity must be treated as failed until PIT/ESS checks.",
                RuntimeWarning,
                stacklevel=2,
            )
    contexts = _audit_context_subset(flow_pack, contexts, seed)
    z_grid = np.arange(-6.0, 7.0, dtype=np.float64)
    probabilities = np.tile(norm.cdf(z_grid), len(contexts))
    repeated_contexts = np.repeat(contexts, len(z_grid), axis=0)
    retry_before = _rqs_retry_count()
    inverse_calls_before = _rqs_inverse_call_count()

    quantiles = scalar_spline_flow_icdf(
        flow_pack,
        probabilities,
        context=repeated_contexts,
        batch_size=8_192,
    ).reshape(len(contexts), len(z_grid))
    if not np.isfinite(quantiles).all():
        raise FloatingPointError(f"{name}: non-finite inverse-CDF values.")
    quantile_steps = np.diff(quantiles, axis=1)
    if not np.all(quantile_steps > 0.0):
        failing = np.argwhere(quantile_steps <= 0.0)
        row, column = map(int, failing[0])
        standardized_context = (
            contexts[row] - flow_pack["context_scaler"].mean
        ) / flow_pack["context_scaler"].std
        raise RuntimeError(
            f"{name}: inverse CDF is not strictly monotone: "
            f"ties={int(np.sum(quantile_steps == 0.0))}, "
            f"decreases={int(np.sum(quantile_steps < 0.0))}, "
            f"minimum step={float(np.min(quantile_steps)):.3e}; "
            f"first failure at z=({z_grid[column]:.1f}, "
            f"{z_grid[column + 1]:.1f}), max |standardized context|="
            f"{float(np.max(np.abs(standardized_context))):.3f}."
        )

    recovered_probability = scalar_spline_flow_cdf(
        flow_pack,
        quantiles.reshape(-1, 1),
        context=repeated_contexts,
        batch_size=8_192,
    )
    recovered_z = norm.ppf(
        np.clip(
            recovered_probability,
            np.nextafter(0.0, 1.0),
            np.nextafter(1.0, 0.0),
        )
    ).reshape(quantiles.shape)
    z_error = np.abs(recovered_z - z_grid[None, :])
    q99_error = float(np.quantile(z_error, 0.99))
    max_error = float(np.max(z_error))
    if q99_error > 1.0e-4 or max_error > 2.0e-3:
        raise RuntimeError(
            f"{name}: inverse/forward closure failed: "
            f"q99={q99_error:.3g}, max={max_error:.3g}."
        )

    log_density = _flow_log_prob(
        flow_pack,
        quantiles.reshape(-1, 1),
        context=repeated_contexts,
        batch_size=8_192,
    )
    if not np.isfinite(log_density).all():
        raise FloatingPointError(f"{name}: non-finite density at its quantiles.")

    first = _draw_conditional(flow_pack, contexts, 2, seed + 1)
    second = _draw_conditional(flow_pack, contexts, 2, seed + 1)
    if not np.array_equal(first, second):
        raise RuntimeError(f"{name}: repeated seeded draws are not identical.")
    retry_delta = _rqs_retry_count() - retry_before
    inverse_call_delta = _rqs_inverse_call_count() - inverse_calls_before
    if retry_delta:
        raise RuntimeError(
            f"{name}: {retry_delta}/{inverse_call_delta} inverse-RQS "
            "kernel calls needed float64 in the numerical preflight. "
            "Retrain this scalar flow; reserve the fallback for a rare "
            "event in the much larger production draw."
        )
    print(
        f"{name} scalar-flow audit passed: q99 |z_back-z|={q99_error:.2e}, "
        f"max={max_error:.2e}, min quantile step="
        f"{float(np.min(quantile_steps)):.2e}, retried RQS kernels="
        f"{retry_delta}/{inverse_call_delta}."
    )


def _fit_classifier_transform(points):
    points = np.asarray(points, dtype=np.float32)
    center = points.mean(axis=0, dtype=np.float64).astype(np.float32)
    scale = points.std(axis=0, dtype=np.float64).astype(np.float32)
    scale = np.where(scale > 1.0e-6, scale, 1.0).astype(np.float32)
    return center, scale


def _transform_classifier_points(points, center, scale):
    return ((np.asarray(points, dtype=np.float32) - center) / scale).astype(np.float32)


def _make_model(input_dim, n_classes):
    return PlainMulticlassMLP(
        input_dim=input_dim,
        n_classes=n_classes,
        **CLASS_MODEL_CONFIG,
    ).to(device)


def _multiclass_fingerprint(class_points, *, n_classes, seed_base):
    digest = hashlib.sha256()
    configuration = json.dumps(
        {
            "n_classes": int(n_classes),
            "seed_base": int(seed_base),
            "classifier_count": int(CLASSIFIER_ENSEMBLE_SIZE),
            "model": CLASS_MODEL_CONFIG,
            "training": CLASS_TRAINING_CONFIG,
            "loss": "equal_prior_multiclass_ce_only_v9_classifier_ensemble",
            "input_transform": "ordinary_standardization_v1",
        },
        sort_keys=True,
        separators=(",", ":"),
    )
    digest.update(configuration.encode("utf-8"))
    array = np.ascontiguousarray(class_points)
    digest.update(str(array.shape).encode("ascii"))
    digest.update(str(array.dtype).encode("ascii"))
    digest.update(array.view(np.uint8))
    return digest.hexdigest()


def _validation_ce(model, points, labels):
    model.eval()
    batch_size = int(CLASS_TRAINING_CONFIG["validation_batch_size"])
    loss_sum, count = 0.0, 0
    with torch.no_grad():
        for start in range(0, len(points), batch_size):
            stop = start + batch_size
            batch = points[start:stop].to(device)
            target = labels[start:stop].to(device)
            loss = F.cross_entropy(model(batch), target)
            loss_sum += float(loss.cpu()) * len(batch)
            count += len(batch)
    return loss_sum / max(1, count)


def train_multiclass_classifier(
    class_points,
    *,
    n_classes,
    checkpoint_dir,
    seed_base,
):
    # Train equal-prior multiclass cross entropy, and no other term.
    class_points = _assert_finite(
        "class_points", class_points, ndim=3
    ).astype(np.float32)
    if class_points.shape[1] != int(n_classes):
        raise ValueError("One row per class is required in every group.")

    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    fingerprint = _multiclass_fingerprint(
        class_points, n_classes=n_classes, seed_base=seed_base
    )

    split_rng = np.random.default_rng(SEED + 700 + int(n_classes))
    order = split_rng.permutation(len(class_points))
    n_validation = max(
        1, int(CLASS_TRAINING_CONFIG["validation_fraction"] * len(order))
    )
    validation_index, training_index = order[:n_validation], order[n_validation:]

    training_flat = class_points[training_index].reshape(
        -1, class_points.shape[-1]
    )
    validation_flat = class_points[validation_index].reshape(
        -1, class_points.shape[-1]
    )
    center, scale = _fit_classifier_transform(training_flat)
    training_tensor = torch.as_tensor(
        _transform_classifier_points(training_flat, center, scale),
        dtype=torch.float32,
    )
    validation_tensor = torch.as_tensor(
        _transform_classifier_points(validation_flat, center, scale),
        dtype=torch.float32,
    )
    training_labels = torch.arange(int(n_classes), dtype=torch.long).repeat(
        len(training_index)
    )
    validation_labels = torch.arange(int(n_classes), dtype=torch.long).repeat(
        len(validation_index)
    )
    training_dataset = TensorDataset(training_tensor, training_labels)

    classifier_packs = []
    for classifier_index in range(CLASSIFIER_ENSEMBLE_SIZE):
        classifier_seed = int(seed_base + classifier_index)
        checkpoint = checkpoint_dir / f"classifier.member_{classifier_index:02d}.pt"
        if LOAD_IF_AVAILABLE and checkpoint.exists():
            try:
                saved = torch.load(
                    checkpoint, map_location=device, weights_only=False
                )
            except TypeError:
                saved = torch.load(checkpoint, map_location=device)
            if (
                saved.get("fingerprint") != fingerprint
                or int(saved.get("classifier_index", -1)) != classifier_index
                or int(saved.get("classifier_seed", -1)) != classifier_seed
            ):
                raise RuntimeError(
                    f"Checkpoint {checkpoint} belongs to a different CE experiment."
                )
            model = _make_model(class_points.shape[-1], n_classes)
            model.load_state_dict(saved["state_dict"])
            model.eval()
            classifier_packs.append(
                {
                    "model": model,
                    "center": np.asarray(saved["center"], dtype=np.float32),
                    "scale": np.asarray(saved["scale"], dtype=np.float32),
                    "history": saved.get("history", {}),
                    "checkpoint": checkpoint,
                    "classifier_index": classifier_index,
                    "classifier_seed": classifier_seed,
                }
            )
            print("Loaded", checkpoint)
            continue

        set_torch_seed(classifier_seed)
        generator = torch.Generator().manual_seed(classifier_seed + 120_000)
        loader = DataLoader(
            training_dataset,
            batch_size=int(CLASS_TRAINING_CONFIG["batch_size"]),
            shuffle=True,
            generator=generator,
        )
        model = _make_model(class_points.shape[-1], n_classes)
        if classifier_index == 0:
            parameter_count = sum(p.numel() for p in model.parameters())
            print(f"Plain {n_classes}-class MLP parameters/member: {parameter_count:,}")
            print(f"Classifier ensemble members: {CLASSIFIER_ENSEMBLE_SIZE}")
            print("Optimized objective: equal-prior multiclass CE only")
        optimizer = torch.optim.Adam(
            model.parameters(), lr=float(CLASS_TRAINING_CONFIG["learning_rate"])
        )
        history = {"train_ce": [], "validation_ce": [], "learning_rate": []}
        best_state, best_value, best_epoch, stale = None, math.inf, 0, 0

        for epoch in range(int(CLASS_TRAINING_CONFIG["n_epochs"])):
            learning_rate = max(
                float(CLASS_TRAINING_CONFIG["min_learning_rate"]),
                float(CLASS_TRAINING_CONFIG["learning_rate"])
                * float(CLASS_TRAINING_CONFIG["lr_scheduler_factor"])
                ** (
                    epoch
                    // int(CLASS_TRAINING_CONFIG["lr_scheduler_step_epochs"])
                ),
            )
            for parameter_group in optimizer.param_groups:
                parameter_group["lr"] = learning_rate
            model.train()
            train_sum, train_count = 0.0, 0
            for batch, labels in loader:
                batch = batch.to(device)
                labels = labels.to(device)
                objective = F.cross_entropy(model(batch), labels)
                if not torch.isfinite(objective):
                    raise FloatingPointError("Non-finite multiclass CE.")
                optimizer.zero_grad(set_to_none=True)
                objective.backward()
                optimizer.step()
                train_sum += float(objective.detach().cpu()) * len(batch)
                train_count += len(batch)

            train_ce = train_sum / max(1, train_count)
            validation_ce = _validation_ce(
                model, validation_tensor, validation_labels
            )
            history["train_ce"].append(train_ce)
            history["validation_ce"].append(validation_ce)
            history["learning_rate"].append(
                float(optimizer.param_groups[0]["lr"])
            )
            if validation_ce < best_value - float(
                CLASS_TRAINING_CONFIG["minimum_improvement"]
            ):
                best_value = validation_ce
                best_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch + 1
                stale = 0
            else:
                stale += 1
            if (
                epoch == 0
                or (epoch + 1)
                % int(CLASS_TRAINING_CONFIG["lr_scheduler_step_epochs"])
                == 0
                or epoch + 1 == int(CLASS_TRAINING_CONFIG["n_epochs"])
            ):
                print(
                    f"classifier {classifier_index + 1:02d}/"
                    f"{CLASSIFIER_ENSEMBLE_SIZE:02d}, epoch {epoch + 1:3d}: "
                    f"train CE={train_ce:.6f}, validation CE={validation_ce:.6f}, "
                    f"lr={learning_rate:.1e}"
                )
            if stale >= int(CLASS_TRAINING_CONFIG["patience"]):
                print(f"classifier: early stopping after {epoch + 1} epochs")
                break

        if best_state is None:
            raise RuntimeError("No finite CE checkpoint was produced.")
        model.load_state_dict(best_state)
        model.eval()
        history["selected"] = [
            {"epoch": best_epoch, "validation_ce": best_value}
        ]
        history["optimized_terms"] = ["multiclass_ce"]
        torch.save(
            {
                "state_dict": model.state_dict(),
                "center": center,
                "scale": scale,
                "history": history,
                "n_classes": int(n_classes),
                "input_dim": int(class_points.shape[-1]),
                "model_config": CLASS_MODEL_CONFIG,
                "fingerprint": fingerprint,
                "classifier_index": classifier_index,
                "classifier_seed": classifier_seed,
            },
            checkpoint,
        )
        classifier_packs.append(
            {
                "model": model,
                "center": center,
                "scale": scale,
                "history": history,
                "checkpoint": checkpoint,
                "classifier_index": classifier_index,
                "classifier_seed": classifier_seed,
            }
        )
    return classifier_packs


@torch.no_grad()
def predict_class_probabilities(classifier_packs, points, batch_size=65_536):
    """Proxy probabilities encoding arithmetic means of member ratios.

    Every downstream correction uses class 0 as its numerator.  For each
    alternative class j, this routine first computes the member-wise
    positive softmax quotient d_0/d_j and then averages those quotients.
    The returned normalized proxy has exactly those averaged quotients,
    so the existing probability-ratio interface remains unchanged.
    """
    if not classifier_packs:
        raise RuntimeError("At least one classifier is required.")
    points = _assert_finite("prediction points", points)
    original_shape = points.shape[:-1]
    flat = points.reshape(-1, points.shape[-1]).astype(np.float32)
    tiny = np.finfo(np.float64).tiny
    chunks = []
    for start in range(0, len(flat), int(batch_size)):
        stop = start + int(batch_size)
        ratio_mean = None
        for pack in classifier_packs:
            transformed = _transform_classifier_points(
                flat[start:stop], pack["center"], pack["scale"]
            )
            tensor = torch.as_tensor(
                transformed, dtype=torch.float32, device=device
            )
            probabilities = (
                torch.softmax(pack["model"](tensor).to(torch.float64), dim=1)
                .detach()
                .cpu()
                .numpy()
            )
            member_ratios = probabilities[:, :1] / np.maximum(
                probabilities[:, 1:], tiny
            )
            if ratio_mean is None:
                ratio_mean = member_ratios / float(len(classifier_packs))
            else:
                ratio_mean += member_ratios / float(len(classifier_packs))
        if ratio_mean is None or not np.isfinite(ratio_mean).all():
            raise FloatingPointError("Classifier ensemble returned non-finite ratios.")
        scores = np.concatenate(
            [np.ones((len(ratio_mean), 1)), 1.0 / np.maximum(ratio_mean, tiny)],
            axis=1,
        )
        scores /= np.max(scores, axis=1, keepdims=True)
        chunks.append(scores / scores.sum(axis=1, keepdims=True))
    probabilities = np.concatenate(chunks, axis=0)
    return probabilities.reshape(*original_shape, probabilities.shape[-1])


@torch.no_grad()
def predict_class_log_probabilities(classifier_packs, points, batch_size=65_536):
    points = _assert_finite("log-probability points", points)
    probabilities = predict_class_probabilities(
        classifier_packs, points, batch_size=batch_size
    )
    log_probabilities = np.log(
        np.maximum(probabilities, np.finfo(np.float64).tiny)
    )
    if not np.isfinite(log_probabilities).all():
        raise FloatingPointError("Classifier ensemble returned non-finite log ratios.")
    return log_probabilities


def class_probability_ratio(probabilities, numerator, denominator):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    denominator_probability = np.maximum(
        probabilities[..., int(denominator)], np.finfo(np.float64).tiny
    )
    return probabilities[..., int(numerator)] / denominator_probability


def class_log_ratio(log_probabilities, numerator, denominator):
    log_probabilities = np.asarray(log_probabilities, dtype=np.float64)
    return (
        log_probabilities[..., int(numerator)]
        - log_probabilities[..., int(denominator)]
    )


def normalized_probability_ratios(ratios, axis=-1):
    ratios = np.asarray(ratios, dtype=np.float64)
    total = np.sum(ratios, axis=axis, keepdims=True)
    if not np.all(np.isfinite(total)) or np.any(total <= 0.0):
        raise FloatingPointError("Invalid softmax-probability ratio mass.")
    return ratios / total


def normalized_log_weights(log_weights):
    # Stable normalization for generic density log weights.
    tensor = torch.as_tensor(log_weights, dtype=torch.float64)
    return torch.softmax(tensor, dim=0).cpu().numpy()


def probability_floor_fraction(probabilities, class_index, threshold=1.0e-12):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    return float(np.mean(probabilities[..., int(class_index)] < threshold))
def select_empirical_context_slices(context_pool):
    context_pool = _assert_finite("context pool", context_pool, ndim=2).astype(np.float32)
    center = np.median(context_pool, axis=0)
    mad = 1.4826 * np.median(np.abs(context_pool - center), axis=0)
    scale = np.where(mad > 1.0e-6, mad, context_pool.std(axis=0))
    scale = np.where(scale > 1.0e-6, scale, 1.0)
    score = np.max(np.abs((context_pool - center) / scale), axis=1)
    probabilities = np.array([0.50, 0.90, 0.97, 0.99, 0.995, 0.999, 0.9998, 1.0])
    targets = np.quantile(score, probabilities)
    indices = np.array([int(np.argmin(np.abs(score - value))) for value in targets])
    return context_pool[indices], score[indices], probabilities


def sample_only_flow_audit(flow_pack, contexts, truth_draws, scores, probabilities, name, seed):
    truth_draws = _assert_finite("simulator audit draws", truth_draws, ndim=3)
    flow_draws = _draw_conditional(flow_pack, contexts, truth_draws.shape[1], seed)
    rows = []
    quantiles = np.array([0.001, 0.01, 0.05, 0.50, 0.95, 0.99, 0.999])
    for index, (truth, learned) in enumerate(zip(truth_draws, flow_draws)):
        truth_scale = np.maximum(truth.std(axis=0), 1.0e-6)
        coordinate_w1 = np.array([
            wasserstein_distance(truth[:, feature], learned[:, feature])
            for feature in range(truth.shape[1])
        ]) / truth_scale
        mean_error = np.abs(learned.mean(axis=0) - truth.mean(axis=0)) / truth_scale
        quantile_error = np.max(np.abs(
            np.quantile(learned, quantiles, axis=0)
            - np.quantile(truth, quantiles, axis=0)
        ) / truth_scale, axis=0)

        projection_rng = np.random.default_rng(seed + 10_000 + index)
        directions = projection_rng.normal(size=(32, truth.shape[1]))
        directions /= np.linalg.norm(directions, axis=1, keepdims=True)
        truth_projection = truth @ directions.T
        learned_projection = learned @ directions.T
        projection_scale = np.maximum(truth_projection.std(axis=0), 1.0e-6)
        sliced_w1 = np.array([
            wasserstein_distance(truth_projection[:, direction], learned_projection[:, direction])
            for direction in range(len(directions))
        ]) / projection_scale

        truth_correlation = np.corrcoef(truth, rowvar=False)
        learned_correlation = np.corrcoef(learned, rowvar=False)
        correlation_error = np.max(np.abs(truth_correlation - learned_correlation))
        rows.append({
            "flow": name,
            "context quantile": float(probabilities[index]),
            "context score": float(scores[index]),
            "max coordinate W1 / truth sigma": float(np.max(coordinate_w1)),
            "q95 sliced W1 / truth sigma": float(np.quantile(sliced_w1, 0.95)),
            "max mean error / truth sigma": float(np.max(mean_error)),
            "max tail-quantile error / truth sigma": float(np.max(quantile_error)),
            "max correlation error": float(correlation_error),
        })
    result = pd.DataFrame(rows)
    if not np.isfinite(result.select_dtypes(include=[np.number])).all().all():
        raise FloatingPointError(f"{name}: non-finite sample-only flow audit.")
    print(
        f"{name} sample-only joint-tail audit: worst q95 sliced W1="
        f"{result['q95 sliced W1 / truth sigma'].max():.3f}."
    )
    return result


# Part I — nuisance marginalized implicitly

The nuisance parameter is drawn from its design density inside the simulator and then omitted.  We train

$$q_P^m(\mu\mid x),\qquad q_L^m(x\mid\mu),$$

and construct three balanced classes

$$
\Pi_S=\rho_\mu(\mu)p_m(x\mid\mu),\quad
\Pi_P=m_\rho(x)q_P^m(\mu\mid x),\quad
\Pi_L=\rho_\mu(\mu)q_L^m(x\mid\mu).
$$

Part I retains the validated scalar conditional spline for $q_P^m$, because a one-dimensional target is not naturally represented by alternating coupling layers.  The likelihood proposal uses the original Exercise-9 spline architecture.  Both are trained on ordinary independent design simulations, without tail-stratified training or density-aware coordinates.


In [ ]:
# Independent simulations for the two frozen proposals.
rng = np.random.default_rng(SEED + 10)
theta_qp = sample_design(N_FLOW, rng)
x_qp = simulate(theta_qp, rng)
set_torch_seed(SEED + 11)
q_p = train_spline_flow(
    theta_qp[:, :1],
    context=x_qp,
    checkpoint=MODEL_DIR / "q_p_marginal_mu_given_x.pt",
    model_config=SCALAR_FLOW_MODEL_CONFIG,
    training_config=SCALAR_FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 11,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_qp, x_qp

rng = np.random.default_rng(SEED + 20)
theta_qlm = sample_design(N_FLOW, rng)
# Only mu is retained as context; alpha is marginalized through simulation.
x_qlm = simulate(theta_qlm, rng)
set_torch_seed(SEED + 21)
q_lm = train_spline_flow(
    x_qlm,
    context=theta_qlm[:, :1],
    checkpoint=MODEL_DIR / "q_lm_x_given_mu.pt",
    model_config=FLOW_MODEL_CONFIG,
    training_config=FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 21,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_qlm, x_qlm

# Sample-only q_L^m audit at central through extreme empirical mu ranks.
rng = np.random.default_rng(SEED + 22)
qlm_context_pool = sample_design(50_000 if not SMOKE_MODE else 2_000, rng)[:, :1]
qlm_audit_contexts, qlm_audit_scores, qlm_audit_probabilities = (
    select_empirical_context_slices(qlm_context_pool)
)
n_audit_contexts = len(qlm_audit_contexts)
mu_truth = np.repeat(qlm_audit_contexts[:, 0], N_FLOW_AUDIT_SAMPLES)
alpha_truth = sample_alpha(n_audit_contexts * N_FLOW_AUDIT_SAMPLES, rng)
qlm_truth_draws = simulate(
    np.column_stack([mu_truth, alpha_truth]), rng
).reshape(n_audit_contexts, N_FLOW_AUDIT_SAMPLES, 3)
qlm_tail_audit = sample_only_flow_audit(
    q_lm,
    qlm_audit_contexts,
    qlm_truth_draws,
    qlm_audit_scores,
    qlm_audit_probabilities,
    r"$q_L^m(x\mid\mu)$",
    SEED + 23,
)
display(qlm_tail_audit.style.format(precision=4))

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### Preflight: invertibility before building classifier classes

This audit uses independent ordinary and extreme defensive-design contexts.  Before inversion, it verifies that the explicit target anchors lie inside the learned standardized spline domain.  The scalar flow must then have strict quantile monotonicity, inverse/forward closure, finite log density, and exact repeated-seed sampling, with **zero** precision retries on this compact preflight.  The thresholds catch a support mismatch, stiff inverse, or collapsed inverse rather than grade density accuracy.  A failed float64 retry stops here; it is never converted into rejection sampling.  Proposal fidelity is assessed later by held-out closure, PIT, ESS, Pareto-$k$, and sample-only audits.


In [ ]:
rng = np.random.default_rng(SEED + 90)
theta_qp_audit = sample_design(16_384, rng)
theta_qp_tail = np.column_stack([
    np.array([-18, -15, -12, -10, 10, 12, 15, 18], dtype=np.float32),
    np.array([0, 6, -4, 3, -3, 4, -6, 0], dtype=np.float32),
])
x_qp_audit = np.concatenate([
    simulate(theta_qp_audit, rng),
    simulate(theta_qp_tail, rng),
])
audit_scalar_target_support(
    q_p, theta_qp_tail[:, :1], r"$q_P(\mu\mid x)$"
)
audit_scalar_flow(q_p, x_qp_audit, r"$q_P(\mu\mid x)$", SEED + 91)
del theta_qp_audit, theta_qp_tail, x_qp_audit


In [ ]:
def build_three_class_groups(n_groups, q_p, q_lm, seed):
    rng = np.random.default_rng(seed)
    theta_s = sample_design(n_groups, rng)
    mu_s = theta_s[:, :1]
    x_s = simulate(theta_s, rng)
    mu_p = _draw_conditional(q_p, x_s, 1, seed + 1)[:, 0, :]
    x_l = _draw_conditional(q_lm, mu_s, 1, seed + 2)[:, 0, :]
    points_s = np.column_stack([mu_s, x_s])
    points_p = np.column_stack([mu_p, x_s])
    points_l = np.column_stack([mu_s, x_l])
    groups = np.stack([points_s, points_p, points_l], axis=1).astype(np.float32)
    return _assert_finite("three-class groups", groups, ndim=3)

three_retry_before = _rqs_retry_count()
three_class_groups = build_three_class_groups(
    N_CLASS, q_p, q_lm, SEED + 100
)
print("Three-class grouped tensor:", three_class_groups.shape)
print(
    "Inverse-RQS kernel calls retried in float64 during three-class construction:",
    _rqs_retry_count() - three_retry_before,
)


## Post-training conditional normalization and bridge check

From the softmax probabilities we form

$$r_P^m=\frac{d_S}{d_P},\qquad r_L^m=\frac{d_S}{d_L}.$$

The post-training conditional masses are

$$Z_P^m(x)=\mathbb E_{q_P^m}[r_P^m],\qquad
Z_L^m(\mu)=\mathbb E_{q_L^m}[r_L^m].$$

They are estimated with fresh proposal draws and used only as finite-model normalization corrections.  The normalized bridge

$$
\ell_B(\mu,x)=\log\rho_\mu+\log q_L^m-\log q_P^m
+\log d_P-\log d_L+\log Z_P^m-\log Z_L^m
$$

should equal $\log m_\rho(x)$ and therefore be independent of $\mu$ at fixed $x$.  It is evaluated only after training and remains a consistency diagnostic rather than a loss.


In [ ]:
def build_three_bridge_bundle(n_groups, n_inner, n_mass_inner, q_p, q_lm, seed):
    if int(n_inner) < 2:
        raise ValueError("Bridge variance requires at least two draws per anchor.")
    rng = np.random.default_rng(seed)
    theta_anchor = sample_tail_enriched_design(n_groups, rng)
    x_anchor = simulate(theta_anchor, rng)
    mu = _draw_conditional(q_p, x_anchor, n_inner, seed + 1)
    x_repeat = np.repeat(x_anchor[:, None, :], n_inner, axis=1)
    points = np.concatenate([mu, x_repeat], axis=2).astype(np.float32)
    flat_mu = mu.reshape(-1, 1)
    flat_x = x_repeat.reshape(-1, 3)
    bridge_base = (
        design_mu_logpdf(flat_mu[:, 0])
        + _flow_log_prob(q_lm, flat_x, context=flat_mu)
        - _flow_log_prob(q_p, flat_mu, context=flat_x)
    ).reshape(n_groups, n_inner).astype(np.float32)

    # The exact finite-model bridge contains -log Z_L(mu).  Estimate it
    # with an independent nested q_L bank after training; log Z_P(x) is
    # constant along this fixed-x diagnostic variance.
    x_zl = _draw_conditional(q_lm, flat_mu, n_mass_inner, seed + 2)
    mu_zl = np.repeat(flat_mu[:, None, :], n_mass_inner, axis=1)
    bridge_zl_points = np.concatenate([mu_zl, x_zl], axis=2).reshape(
        n_groups, n_inner, n_mass_inner, 4
    ).astype(np.float32)
    return {
        "bridge_points": _assert_finite("three bridge points", points),
        "bridge_base": _assert_finite("three bridge base", bridge_base),
        "bridge_zl_points": _assert_finite("three bridge ZL points", bridge_zl_points),
    }



## Pure-CE three-class training

The model below optimizes one and only one quantity: held-out equal-prior multiclass cross entropy.  In particular, the previous direct-$a$ binary term and its $-\log 2$ offset have been removed completely.


In [ ]:
three_ce = train_multiclass_classifier(
    three_class_groups,
    n_classes=3,
    checkpoint_dir=MODEL_DIR / "part1_marginal_three_class_ce",
    seed_base=SEED + 400,
)
del three_class_groups
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Independent closure of the marginalized experiment

Analytic truth is now used only to test the frozen flow-plus-CE construction.  Raw and post-hoc-normalized likelihood routes are shown separately, and fresh banks test conditional masses, bridge consistency, calibration, and importance tails.


In [ ]:
def three_zp(classifier_packs, x_values, n_reference, seed):
    x_values = np.atleast_2d(np.asarray(x_values, dtype=np.float32))
    mu = _draw_conditional(q_p, x_values, n_reference, seed)
    points = np.concatenate(
        [mu, np.repeat(x_values[:, None, :], n_reference, axis=1)], axis=2
    )
    probabilities = predict_class_probabilities(classifier_packs, points)
    return np.mean(class_probability_ratio(probabilities, 0, 1), axis=1)


def three_log_zp(classifier_packs, x_values, n_reference, seed):
    return np.log(three_zp(classifier_packs, x_values, n_reference, seed))


def three_zl(classifier_packs, mu_values, n_reference, seed):
    mu_values = np.asarray(mu_values, dtype=np.float32).reshape(-1, 1)
    x = _draw_conditional(q_lm, mu_values, n_reference, seed)
    points = np.concatenate(
        [np.repeat(mu_values[:, None, :], n_reference, axis=1), x], axis=2
    )
    probabilities = predict_class_probabilities(classifier_packs, points)
    return np.mean(class_probability_ratio(probabilities, 0, 2), axis=1)


def three_log_zl(classifier_packs, mu_values, n_reference, seed):
    return np.log(three_zl(classifier_packs, mu_values, n_reference, seed))


def three_closure(classifier_packs, mu_grid, x_observed, n_reference, seed):
    mu_grid = np.asarray(mu_grid, dtype=float)
    x_grid = np.repeat(np.asarray(x_observed)[None, :], len(mu_grid), axis=0)
    points = np.column_stack([mu_grid, x_grid]).astype(np.float32)
    log_probabilities = predict_class_log_probabilities(classifier_packs, points)
    log_rp = class_log_ratio(log_probabilities, 0, 1)
    log_rl = class_log_ratio(log_probabilities, 0, 2)
    log_qp = _flow_log_prob(q_p, mu_grid[:, None], context=x_grid)
    log_ql = _flow_log_prob(q_lm, x_grid, context=mu_grid[:, None])
    log_zp = float(three_log_zp(classifier_packs, x_observed, n_reference, seed)[0])
    log_zl = three_log_zl(classifier_packs, mu_grid, n_reference, seed + 1)

    posterior_route, _ = normalize_log_curve(
        log_qp + log_rp - log_zp, mu_grid
    )
    raw_likelihood_posterior, _ = normalize_log_curve(
        design_mu_logpdf(mu_grid) + log_ql + log_rl, mu_grid
    )
    likelihood_posterior, _ = normalize_log_curve(
        design_mu_logpdf(mu_grid) + log_ql + log_rl - log_zl, mu_grid
    )
    raw_bridge = (
        design_mu_logpdf(mu_grid) + log_ql - log_qp
        + log_probabilities[:, 1] - log_probabilities[:, 2]
    )
    normalized_bridge = raw_bridge + log_zp - log_zl
    return {
        "posterior": posterior_route,
        "raw_likelihood_posterior": raw_likelihood_posterior,
        "likelihood_posterior": likelihood_posterior,
        "raw_log_evidence_consistency": raw_bridge,
        "log_evidence_consistency": normalized_bridge,
        "log_zp": log_zp,
        "log_zl": log_zl,
        "log_ratio_grid": log_rp,
    }


MU_GRID = np.linspace(-3.8, 3.8, 321)
truth_mu, _ = normalize_log_curve(
    design_mu_logpdf(MU_GRID) + marginal_log_likelihood(X_OBS, MU_GRID),
    MU_GRID,
)
MU_EVIDENCE_GRID = np.linspace(-10.0, 10.0, 4001)
_, LOG_EVIDENCE_TRUTH = normalize_log_curve(
    design_mu_logpdf(MU_EVIDENCE_GRID)
    + marginal_log_likelihood(X_OBS, MU_EVIDENCE_GRID),
    MU_EVIDENCE_GRID,
)
x_grid_part1 = np.repeat(X_OBS[None, :], len(MU_GRID), axis=0)
q_p_curve, _ = normalize_log_curve(
    _flow_log_prob(q_p, MU_GRID[:, None], context=x_grid_part1), MU_GRID
)
q_lm_curve, _ = normalize_log_curve(
    design_mu_logpdf(MU_GRID)
    + _flow_log_prob(q_lm, x_grid_part1, context=MU_GRID[:, None]),
    MU_GRID,
)
closure_ce = three_closure(
    three_ce, MU_GRID, X_OBS, N_DIAGNOSTIC_REFERENCE, SEED + 500
)

rng = np.random.default_rng(SEED + 510)
n_context_check = 36 if SMOKE_MODE else (80 if FAST_MODE else 160)
theta_check = sample_design(n_context_check, rng)
x_check = simulate(theta_check, rng)
mu_check = sample_mu(n_context_check, rng)
heldout_norm = {
    "log_zp": three_log_zp(
        three_ce, x_check, N_DIAGNOSTIC_REFERENCE, SEED + 520
    ),
    "log_zl": three_log_zl(
        three_ce, mu_check, N_DIAGNOSTIC_REFERENCE, SEED + 521
    ),
}

mu_tail = _draw_conditional(
    q_p,
    X_OBS[None, :],
    2_000 if SMOKE_MODE else (20_000 if FAST_MODE else 80_000),
    SEED + 530,
)[0]
x_tail = np.repeat(X_OBS[None, :], len(mu_tail), axis=0)
tail_points = np.column_stack([mu_tail, x_tail])
tail_probabilities = predict_class_probabilities(three_ce, tail_points)
tail_log_probabilities = np.log(
    np.maximum(tail_probabilities, np.finfo(np.float64).tiny)
)
tail_ratios = class_probability_ratio(tail_probabilities, 0, 1)
tail_log_weights = class_log_ratio(tail_log_probabilities, 0, 1)
tail_summary = importance_tail_summary(tail_log_weights)

mu_likelihood_tail = float(MU_GRID[np.argmax(truth_mu)])
x_likelihood_tail = _draw_conditional(
    q_lm, [[mu_likelihood_tail]], len(mu_tail), SEED + 531
)[0]
likelihood_tail_points = np.column_stack(
    [np.full(len(x_likelihood_tail), mu_likelihood_tail), x_likelihood_tail]
)
likelihood_tail_probabilities = predict_class_probabilities(
    three_ce, likelihood_tail_points
)
likelihood_tail_log_probabilities = np.log(
    np.maximum(likelihood_tail_probabilities, np.finfo(np.float64).tiny)
)
likelihood_tail_summary = importance_tail_summary(
    class_log_ratio(likelihood_tail_log_probabilities, 0, 2)
)

bridge_three_audit = [
    build_three_bridge_bundle(
        N_BRIDGE_AUDIT_GROUPS,
        N_BRIDGE_AUDIT_INNER,
        N_BRIDGE_AUDIT_MASS_INNER,
        q_p,
        q_lm,
        SEED + 120_000 + 100 * bank,
    )
    for bank in range(N_BRIDGE_AUDIT_BANKS)
]


def heldout_three_bridge_rms(classifier_packs, banks):
    values = []
    for bundle in banks:
        log_probabilities = predict_class_log_probabilities(
            classifier_packs, bundle["bridge_points"]
        )
        zl_probabilities = predict_class_probabilities(
            classifier_packs, bundle["bridge_zl_points"]
        )
        log_zl = np.log(
            np.mean(class_probability_ratio(zl_probabilities, 0, 2), axis=2)
        )
        implied = (
            bundle["bridge_base"]
            + log_probabilities[..., 1]
            - log_probabilities[..., 2]
            - log_zl
        )
        values.append(np.std(implied, axis=1, ddof=1))
    return np.concatenate(values)


heldout_bridge_rms = heldout_three_bridge_rms(three_ce, bridge_three_audit)
del bridge_three_audit

support_mask = truth_mu > 1.0e-4 * truth_mu.max()
consistency_delta = closure_ce["log_evidence_consistency"] - LOG_EVIDENCE_TRUTH
all_log_z = np.concatenate([heldout_norm["log_zp"], heldout_norm["log_zl"]])
three_summary = pd.DataFrame(
    [
        {
            "training objective": THREE_CE_LABEL,
            "posterior IAE": integrated_absolute_error(
                truth_mu, closure_ce["posterior"], MU_GRID
            ),
            "raw likelihood-route IAE": integrated_absolute_error(
                truth_mu, closure_ce["raw_likelihood_posterior"], MU_GRID
            ),
            "normalized likelihood IAE": integrated_absolute_error(
                truth_mu, closure_ce["likelihood_posterior"], MU_GRID
            ),
            "evidence RMS (supported)": float(
                np.sqrt(np.mean(consistency_delta[support_mask] ** 2))
            ),
            "held-out bridge median RMS": float(np.median(heldout_bridge_rms)),
            "held-out bridge q95 RMS": float(
                np.quantile(heldout_bridge_rms, 0.95)
            ),
            "RMS log Z": float(np.sqrt(np.mean(all_log_z**2))),
            "q95 |log Z|": float(np.quantile(np.abs(all_log_z), 0.95)),
            "P-class probability floor": probability_floor_fraction(
                tail_probabilities, 1
            ),
            "L-class probability floor": probability_floor_fraction(
                likelihood_tail_probabilities, 2
            ),
            "posterior ESS fraction": tail_summary["ESS_fraction"],
            "posterior Pareto k": tail_summary["pareto_k"],
            "likelihood ESS fraction": likelihood_tail_summary["ESS_fraction"],
            "likelihood Pareto k": likelihood_tail_summary["pareto_k"],
        }
    ]
)
display(three_summary.style.format(precision=4))


In [ ]:
DIRECT_COLOR = "#D55E00"
NORM_COLOR = "#0072B2"
POSTHOC_COLOR = "#009E73"
JOINT_CE_COLOR = "#6A3D9A"

fig, axes = plt.subplots(2, 2, figsize=(12.4, 8.9), constrained_layout=True)
axes[0, 0].plot(MU_GRID, truth_mu, color="black", lw=2.4, label="analytic truth")
axes[0, 0].plot(MU_GRID, q_p_curve, color="0.6", lw=1.5, ls=":", label=r"proposal $q_P$")
axes[0, 0].plot(
    MU_GRID, q_lm_curve, color="0.45", lw=1.4, ls="-.",
    label=r"proposal $\rho_\mu q_L^m$",
)
axes[0, 0].plot(
    MU_GRID, closure_ce["posterior"], color=DIRECT_COLOR, lw=2.0,
    label="CE posterior route",
)
axes[0, 0].plot(
    MU_GRID, closure_ce["raw_likelihood_posterior"], color=NORM_COLOR,
    lw=1.4, ls=":", label="CE likelihood route, raw",
)
axes[0, 0].plot(
    MU_GRID, closure_ce["likelihood_posterior"], color=POSTHOC_COLOR,
    lw=1.8, ls="--", label="same CE model, post-hoc normalized",
)
axes[0, 0].set(xlabel=r"$\mu$", ylabel="posterior density", title="(a) Posterior closure")
axes[0, 0].legend(fontsize=8)

supported = truth_mu > 1.0e-4 * truth_mu.max()
raw_residual = closure_ce["raw_log_evidence_consistency"] - LOG_EVIDENCE_TRUTH
normalized_residual = closure_ce["log_evidence_consistency"] - LOG_EVIDENCE_TRUTH
axes[0, 1].plot(MU_GRID, np.where(supported, raw_residual, np.nan), color="0.55", lw=1.4, ls=":", label="raw CE bridge")
axes[0, 1].plot(MU_GRID, np.where(supported, normalized_residual, np.nan), color=DIRECT_COLOR, lw=2, label="post-hoc-normalized bridge")
axes[0, 1].axhline(0.0, color="black", lw=1)
axes[0, 1].set(
    xlabel=r"$\mu$", ylabel=r"$\log\widehat m-\log m_{\rm true}$",
    title="(b) Fresh normalized evidence bridge",
)
axes[0, 1].legend(fontsize=8)

values_p = heldout_norm["log_zp"]
values_l = heldout_norm["log_zl"]
rng_plot = np.random.default_rng(SEED + 1)
axes[1, 0].scatter(
    rng_plot.normal(0, 0.012, len(values_p)), values_p,
    s=12, alpha=0.42, color=DIRECT_COLOR,
)
axes[1, 0].scatter(
    1.0 + rng_plot.normal(0, 0.012, len(values_l)), values_l,
    s=12, alpha=0.42, color=DIRECT_COLOR, label=THREE_CE_LABEL,
)
axes[1, 0].axhline(0.0, color="black", lw=1)
axes[1, 0].set(
    xticks=[0, 1], xticklabels=[r"$\log Z_P(x)$", r"$\log Z_L(\mu)$"],
    ylabel="held-out conditional log normalizer", title="(c) Independent mass closure",
)
axes[1, 0].legend(fontsize=8)

weights = tail_ratios / np.mean(tail_ratios)
ordered = np.sort(np.maximum(weights, np.finfo(float).tiny))
survival = 1.0 - np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
axes[1, 1].plot(ordered, survival, color=DIRECT_COLOR, lw=1.8, label=THREE_CE_LABEL)
axes[1, 1].set(
    xscale="log", yscale="log", xlabel=r"normalized correction $r_P/\bar r_P$",
    ylabel="empirical survival", title="(d) Posterior-correction tail",
)
axes[1, 1].legend(fontsize=8)
for ax in axes.flat:
    ax.grid(alpha=0.25)
export_exercise9_multiclass_figure(fig, "three_class_ce_posthoc_consistency")
plt.show()


**Figure 1 interpretation.**  One frozen CE classifier supplies every learned curve.  The raw versus post-hoc-normalized likelihood routes show what the conditional $Z_L$ correction changes after training; the bridge panel is a consistency audit, not an optimized quantity.


## Held-out calibration with the nuisance marginalized

Fresh simulator draws test the amortized $\mu$ posterior.  The corrected PIT uses the direct softmax probability ratio $d_S/d_P$ as the importance weight; no logit is exponentiated.


In [ ]:
rng = np.random.default_rng(SEED + 1300)
theta_calibration = sample_design(N_CALIBRATION_CONTEXTS, rng)
x_calibration = simulate(theta_calibration, rng)
mu_calibration = _draw_conditional(
    q_p, x_calibration, N_CALIBRATION_SAMPLES, SEED + 1301
)[..., 0]
calibration_points = np.concatenate(
    [
        mu_calibration[..., None],
        np.repeat(x_calibration[:, None, :], N_CALIBRATION_SAMPLES, axis=1),
    ],
    axis=2,
)
below_truth = mu_calibration <= theta_calibration[:, 0, None]

pit_values = {"proposal q_P": below_truth.mean(axis=1)}
probabilities = predict_class_probabilities(three_ce, calibration_points)
ratios = class_probability_ratio(probabilities, 0, 1)
weights = normalized_probability_ratios(ratios, axis=1)
pit_values[THREE_CE_LABEL] = np.sum(weights * below_truth, axis=1)

NOMINAL_COVERAGE = np.linspace(0.05, 0.95, 19)
coverage_values = {
    name: np.array(
        [
            np.mean(
                (pit >= 0.5 * (1.0 - level))
                & (pit <= 0.5 * (1.0 + level))
            )
            for level in NOMINAL_COVERAGE
        ]
    )
    for name, pit in pit_values.items()
}
calibration_rows = []
for name, pit in pit_values.items():
    ordered = np.sort(pit)
    uniform_quantiles = (np.arange(len(ordered)) + 0.5) / len(ordered)
    calibration_rows.append(
        {
            "method": name,
            "PIT KS distance": float(
                np.max(np.abs(ordered - uniform_quantiles))
            ),
            "max coverage error": float(
                np.max(np.abs(coverage_values[name] - NOMINAL_COVERAGE))
            ),
        }
    )
display(pd.DataFrame(calibration_rows).style.format(precision=4))

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.3), constrained_layout=True)
calibration_colors = {"proposal q_P": "0.55", THREE_CE_LABEL: DIRECT_COLOR}
for name, pit in pit_values.items():
    ordered = np.sort(pit)
    empirical_cdf = np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
    axes[0].plot(
        ordered,
        empirical_cdf,
        lw=1.9,
        color=calibration_colors[name],
        label=name,
    )
    axes[1].plot(
        NOMINAL_COVERAGE,
        coverage_values[name],
        lw=1.9,
        color=calibration_colors[name],
        label=name,
    )
axes[0].plot([0, 1], [0, 1], color="black", ls="--", lw=1)
axes[0].set(
    xlabel="posterior PIT",
    ylabel="empirical CDF",
    title="(a) Held-out posterior PIT",
)
axes[1].plot([0, 1], [0, 1], color="black", ls="--", lw=1)
binomial_sigma = np.sqrt(
    NOMINAL_COVERAGE
    * (1.0 - NOMINAL_COVERAGE)
    / N_CALIBRATION_CONTEXTS
)
axes[1].fill_between(
    NOMINAL_COVERAGE,
    np.maximum(0.0, NOMINAL_COVERAGE - binomial_sigma),
    np.minimum(1.0, NOMINAL_COVERAGE + binomial_sigma),
    color="0.7",
    alpha=0.2,
    linewidth=0,
    label=r"$\pm1\sigma$ binomial",
)
axes[1].set(
    xlabel="nominal equal-tailed coverage",
    ylabel="empirical coverage",
    title="(b) Held-out coverage",
)
for ax in axes:
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
export_exercise9_multiclass_figure(fig, "part1_marginal_heldout_calibration")
plt.show()


# Part II — explicit nuisance with the original joint dual flows

We now retain the full parameter vector $\theta=(\mu,\alpha)$ and train exactly the two proposal types used by the original dual hNPE–hNDE exercise:

$$q_P(\mu,\alpha\mid x),\qquad q_L(x\mid\mu,\alpha).$$

Both use ten conditional rational-quadratic-spline coupling layers, width 512, four residual blocks, 16 bins, tail bound 5, and the original maximum-likelihood training configuration.  They are trained independently on ordinary design simulations.  There is no scalar $q_N$, no hierarchical $q_Pq_N$ factorization, and no four-class construction.

The three balanced joint classes are simply

$$
\Pi_S=\rho(\theta)p(x\mid\theta),\quad
\Pi_P=m_\rho(x)q_P(\theta\mid x),\quad
\Pi_L=\rho(\theta)q_L(x\mid\theta).
$$

Therefore $r_P=d_S/d_P$ corrects the joint posterior flow and $r_L=d_S/d_L$ corrects the joint-parameter likelihood flow.


In [ ]:
q_lm["flow"].to(torch.device("cpu"))
if torch.cuda.is_available():
    torch.cuda.empty_cache()

rng = np.random.default_rng(SEED + 600)
theta_qp_joint = sample_design(N_FLOW, rng)
x_qp_joint = simulate(theta_qp_joint, rng)
set_torch_seed(SEED + 601)
q_p_joint = train_spline_flow(
    theta_qp_joint,
    context=x_qp_joint,
    checkpoint=MODEL_DIR / "q_p_joint_theta_given_x.pt",
    model_config=FLOW_MODEL_CONFIG,
    training_config=FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 601,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_qp_joint, x_qp_joint

rng = np.random.default_rng(SEED + 610)
theta_ql = sample_design(N_FLOW, rng)
x_ql = simulate(theta_ql, rng)
set_torch_seed(SEED + 611)
q_l = train_spline_flow(
    x_ql,
    context=theta_ql,
    checkpoint=MODEL_DIR / "q_l_x_given_theta.pt",
    model_config=FLOW_MODEL_CONFIG,
    training_config=FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 611,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_ql, x_ql

# Sample-only checks at ordinary and empirically extreme contexts.
rng = np.random.default_rng(SEED + 612)
ql_context_pool = sample_design(50_000 if not SMOKE_MODE else 2_000, rng)
ql_audit_contexts, ql_audit_scores, ql_audit_probabilities = (
    select_empirical_context_slices(ql_context_pool)
)
n_audit_contexts = len(ql_audit_contexts)
ql_truth_draws = simulate(
    np.repeat(ql_audit_contexts, N_FLOW_AUDIT_SAMPLES, axis=0), rng
).reshape(n_audit_contexts, N_FLOW_AUDIT_SAMPLES, 3)
ql_tail_audit = sample_only_flow_audit(
    q_l,
    ql_audit_contexts,
    ql_truth_draws,
    ql_audit_scores,
    ql_audit_probabilities,
    r"$q_L(x\mid\mu,\alpha)$",
    SEED + 613,
)
display(ql_tail_audit.style.format(precision=4))

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
def build_joint_three_class_groups(n_groups, q_p_joint, q_l, seed):
    rng = np.random.default_rng(seed)
    theta_s = sample_design(n_groups, rng)
    x_s = simulate(theta_s, rng)
    theta_p = _draw_conditional(
        q_p_joint, x_s, 1, seed + 1
    )[:, 0, :]
    x_l = _draw_conditional(q_l, theta_s, 1, seed + 2)[:, 0, :]
    points_s = np.column_stack([theta_s, x_s])
    points_p = np.column_stack([theta_p, x_s])
    points_l = np.column_stack([theta_s, x_l])
    groups = np.stack(
        [points_s, points_p, points_l], axis=1
    ).astype(np.float32)
    return _assert_finite("joint three-class groups", groups, ndim=3)


joint_retry_before = _rqs_retry_count()
joint_three_class_groups = build_joint_three_class_groups(
    N_CLASS, q_p_joint, q_l, SEED + 620
)
print("Joint three-class grouped tensor:", joint_three_class_groups.shape)
print(
    "Inverse-RQS kernel calls retried during class construction:",
    _rqs_retry_count() - joint_retry_before,
)


## Pure CE for the full $(\mu,\alpha,x)$ classifier

The same plain MLP now maps the five sampled coordinates $(\mu,\alpha,x_1,x_2,x_3)$ directly to three logits.  The only loss is equal-prior multiclass CE.  There are no structured heads and no extra $\mathcal L_a-\log2$ term.


In [ ]:
joint_ce = train_multiclass_classifier(
    joint_three_class_groups,
    n_classes=3,
    checkpoint_dir=MODEL_DIR / "part2_joint_three_class_ce",
    seed_base=SEED + 900,
)
del joint_three_class_groups
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Joint posterior and likelihood closure

The two corrected routes are

$$
\widehat p_H(\theta\mid x)=q_P(\theta\mid x)\,
\frac{d_S/d_P}{Z_P(x)},
$$

$$
\widehat L_H(x\mid\theta)=q_L(x\mid\theta)\,
\frac{d_S/d_L}{Z_L(\theta)}.
$$

The conditional masses are estimated after training from fresh $q_P$ and $q_L$ samples.  The raw proposal routes, raw CE-corrected likelihood route, and post-hoc-normalized corrected routes are all shown, so the work performed by the classifier and by normalization remain separately visible.

The likelihood normalization is deliberately evaluated pointwise: every displayed $\theta$ grid point requires `N_GRID_REFERENCE` fresh draws from $q_L(x\mid\theta)$.  This is the dominant cost of the two-dimensional closure cell.  `N_GRID_REFERENCE` is configurable near the top of the notebook; lowering it makes a faster but noisier diagnostic and does not change training.


In [ ]:
def joint_zp(classifier_packs, x_values, n_reference, seed):
    x_values = np.atleast_2d(np.asarray(x_values, dtype=np.float32))
    theta = _draw_conditional(q_p_joint, x_values, n_reference, seed)
    points = np.concatenate(
        [
            theta,
            np.repeat(x_values[:, None, :], n_reference, axis=1),
        ],
        axis=2,
    )
    probabilities = predict_class_probabilities(classifier_packs, points)
    return np.mean(class_probability_ratio(probabilities, 0, 1), axis=1)


def joint_log_zp(classifier_packs, x_values, n_reference, seed):
    return np.log(joint_zp(classifier_packs, x_values, n_reference, seed))


def joint_zl(classifier_packs, theta_values, n_reference, seed):
    theta_values = np.atleast_2d(
        np.asarray(theta_values, dtype=np.float32)
    )
    context_batch = max(1, 65_536 // int(n_reference))
    masses = []
    for start in range(0, len(theta_values), context_batch):
        theta_chunk = theta_values[start : start + context_batch]
        x = _draw_conditional(
            q_l, theta_chunk, n_reference, seed + start
        )
        points = np.concatenate(
            [
                np.repeat(
                    theta_chunk[:, None, :], n_reference, axis=1
                ),
                x,
            ],
            axis=2,
        )
        probabilities = predict_class_probabilities(classifier_packs, points)
        masses.append(
            np.mean(
                class_probability_ratio(probabilities, 0, 2), axis=1
            )
        )
    return np.concatenate(masses)


def joint_log_zl(classifier_packs, theta_values, n_reference, seed):
    return np.log(joint_zl(classifier_packs, theta_values, n_reference, seed))


def surface_iae(reference, estimate, x_grid, y_grid):
    return float(
        np.trapezoid(
            np.trapezoid(
                np.abs(reference - estimate), y_grid, axis=1
            ),
            x_grid,
        )
    )


MU_GRID_2D = np.linspace(-3.5, 3.5, 181 if not SMOKE_MODE else 71)
ALPHA_GRID_2D = np.linspace(-3.2, 3.2, 161 if not SMOKE_MODE else 61)
MU_MESH, ALPHA_MESH = np.meshgrid(
    MU_GRID_2D, ALPHA_GRID_2D, indexing="ij"
)
THETA_GRID = np.column_stack([MU_MESH.ravel(), ALPHA_MESH.ravel()])
X_GRID = np.repeat(X_OBS[None, :], len(THETA_GRID), axis=0)
POINTS_GRID = np.column_stack([THETA_GRID, X_GRID]).astype(np.float32)

grid_log_probabilities = predict_class_log_probabilities(
    joint_ce, POINTS_GRID
)
log_rp_grid = class_log_ratio(grid_log_probabilities, 0, 1)
log_rl_grid = class_log_ratio(grid_log_probabilities, 0, 2)
log_qp_joint_grid = _flow_log_prob(
    q_p_joint, THETA_GRID, context=X_GRID
)
log_ql_grid = _flow_log_prob(q_l, X_GRID, context=THETA_GRID)
log_zp_observed = float(
    joint_log_zp(
        joint_ce, X_OBS, N_GRID_REFERENCE, SEED + 990
    )[0]
)
log_zl_grid = joint_log_zl(
    joint_ce, THETA_GRID, N_GRID_REFERENCE, SEED + 1000
)

posterior_joint_proposal, _ = normalize_log_surface(
    log_qp_joint_grid.reshape(MU_MESH.shape),
    MU_GRID_2D,
    ALPHA_GRID_2D,
)
posterior_joint_hybrid, _ = normalize_log_surface(
    (log_qp_joint_grid + log_rp_grid - log_zp_observed).reshape(
        MU_MESH.shape
    ),
    MU_GRID_2D,
    ALPHA_GRID_2D,
)
posterior_likelihood_proposal, _ = normalize_log_surface(
    (design_logpdf(THETA_GRID) + log_ql_grid).reshape(MU_MESH.shape),
    MU_GRID_2D,
    ALPHA_GRID_2D,
)
posterior_likelihood_raw, _ = normalize_log_surface(
    (
        design_logpdf(THETA_GRID) + log_ql_grid + log_rl_grid
    ).reshape(MU_MESH.shape),
    MU_GRID_2D,
    ALPHA_GRID_2D,
)
posterior_likelihood_hybrid, _ = normalize_log_surface(
    (
        design_logpdf(THETA_GRID)
        + log_ql_grid
        + log_rl_grid
        - log_zl_grid
    ).reshape(MU_MESH.shape),
    MU_GRID_2D,
    ALPHA_GRID_2D,
)

log_joint_truth = design_logpdf(THETA_GRID) + log_likelihood(
    X_OBS, THETA_GRID
)
posterior_truth_2d, _ = normalize_log_surface(
    log_joint_truth.reshape(MU_MESH.shape),
    MU_GRID_2D,
    ALPHA_GRID_2D,
)

def marginals(surface):
    return (
        np.trapezoid(surface, ALPHA_GRID_2D, axis=1),
        np.trapezoid(surface, MU_GRID_2D, axis=0),
    )


truth_mu_2d, truth_alpha_2d = marginals(posterior_truth_2d)
proposal_mu_2d, proposal_alpha_2d = marginals(
    posterior_joint_proposal
)
hybrid_mu_2d, hybrid_alpha_2d = marginals(posterior_joint_hybrid)
likelihood_raw_mu_2d, likelihood_raw_alpha_2d = marginals(
    posterior_likelihood_raw
)
likelihood_hybrid_mu_2d, likelihood_hybrid_alpha_2d = marginals(
    posterior_likelihood_hybrid
)

joint_summary = pd.DataFrame(
    [
        {
            "route": "raw joint q_P",
            "surface IAE": surface_iae(
                posterior_truth_2d,
                posterior_joint_proposal,
                MU_GRID_2D,
                ALPHA_GRID_2D,
            ),
        },
        {
            "route": "CE-corrected joint q_P",
            "surface IAE": surface_iae(
                posterior_truth_2d,
                posterior_joint_hybrid,
                MU_GRID_2D,
                ALPHA_GRID_2D,
            ),
        },
        {
            "route": "raw rho q_L",
            "surface IAE": surface_iae(
                posterior_truth_2d,
                posterior_likelihood_proposal,
                MU_GRID_2D,
                ALPHA_GRID_2D,
            ),
        },
        {
            "route": "CE-corrected raw rho q_L (before Z_L)",
            "surface IAE": surface_iae(
                posterior_truth_2d,
                posterior_likelihood_raw,
                MU_GRID_2D,
                ALPHA_GRID_2D,
            ),
        },
        {
            "route": "CE-corrected normalized rho q_L",
            "surface IAE": surface_iae(
                posterior_truth_2d,
                posterior_likelihood_hybrid,
                MU_GRID_2D,
                ALPHA_GRID_2D,
            ),
        },
    ]
)
display(joint_summary.style.format(precision=4))


In [ ]:
def highest_density_levels(density, x_grid, y_grid, masses=(0.5, 0.9)):
    density = np.asarray(density, dtype=float)
    cell_mass = density.ravel() * float(np.mean(np.diff(x_grid))) * float(
        np.mean(np.diff(y_grid))
    )
    order = np.argsort(density.ravel())[::-1]
    cumulative = np.cumsum(cell_mass[order])
    levels = []
    for mass in masses:
        index = min(np.searchsorted(cumulative, mass), len(order) - 1)
        levels.append(float(density.ravel()[order[index]]))
    return sorted(levels)


truth_levels = highest_density_levels(
    posterior_truth_2d, MU_GRID_2D, ALPHA_GRID_2D
)
qp_levels = highest_density_levels(
    posterior_joint_hybrid, MU_GRID_2D, ALPHA_GRID_2D
)
ql_levels = highest_density_levels(
    posterior_likelihood_hybrid, MU_GRID_2D, ALPHA_GRID_2D
)
ql_raw_levels = highest_density_levels(
    posterior_likelihood_raw, MU_GRID_2D, ALPHA_GRID_2D
)

fig, axes = plt.subplots(2, 2, figsize=(11.8, 9.0), constrained_layout=True)
axes[0, 0].contour(
    MU_GRID_2D,
    ALPHA_GRID_2D,
    posterior_truth_2d.T,
    levels=truth_levels,
    colors="black",
    linewidths=[2.0, 1.4],
)
axes[0, 0].contour(
    MU_GRID_2D,
    ALPHA_GRID_2D,
    posterior_joint_hybrid.T,
    levels=qp_levels,
    colors=JOINT_CE_COLOR,
    linewidths=[2.0, 1.4],
    linestyles="--",
)
axes[0, 0].set(
    xlabel=r"$\mu$",
    ylabel=r"$\alpha$",
    title="(a) Joint posterior-flow route",
)

axes[0, 1].contour(
    MU_GRID_2D,
    ALPHA_GRID_2D,
    posterior_likelihood_raw.T,
    levels=ql_raw_levels,
    colors=NORM_COLOR,
    linewidths=[1.6, 1.1],
    linestyles=":",
)
axes[0, 1].contour(
    MU_GRID_2D,
    ALPHA_GRID_2D,
    posterior_truth_2d.T,
    levels=truth_levels,
    colors="black",
    linewidths=[2.0, 1.4],
)
axes[0, 1].contour(
    MU_GRID_2D,
    ALPHA_GRID_2D,
    posterior_likelihood_hybrid.T,
    levels=ql_levels,
    colors=POSTHOC_COLOR,
    linewidths=[2.0, 1.4],
    linestyles="--",
)
axes[0, 1].plot([], [], color="black", lw=1.8, label="truth")
axes[0, 1].plot(
    [], [], color=NORM_COLOR, lw=1.5, ls=":", label=r"CE $q_L$, before $Z_L$"
)
axes[0, 1].plot(
    [], [], color=POSTHOC_COLOR, lw=1.7, ls="--", label=r"CE $q_L/Z_L$"
)
axes[0, 1].legend(fontsize=7.5)
axes[0, 1].set(
    xlabel=r"$\mu$",
    ylabel=r"$\alpha$",
    title="(b) Joint likelihood-flow route",
)

axes[1, 0].plot(MU_GRID_2D, truth_mu_2d, color="black", lw=2.2, label="truth")
axes[1, 0].plot(
    MU_GRID_2D,
    likelihood_raw_mu_2d,
    color=NORM_COLOR,
    lw=1.4,
    ls=":",
    label=r"CE $q_L$, before $Z_L$",
)
axes[1, 0].plot(
    MU_GRID_2D,
    proposal_mu_2d,
    color="0.55",
    ls=":",
    label=r"raw $q_P$",
)
axes[1, 0].plot(
    MU_GRID_2D,
    hybrid_mu_2d,
    color=JOINT_CE_COLOR,
    lw=1.9,
    label="CE posterior route",
)
axes[1, 0].plot(
    MU_GRID_2D,
    likelihood_hybrid_mu_2d,
    color=POSTHOC_COLOR,
    lw=1.7,
    ls="--",
    label=r"CE $q_L/Z_L$",
)
axes[1, 0].set(
    xlabel=r"$\mu$", ylabel="posterior density", title="(c) POI marginal"
)

axes[1, 1].plot(
    ALPHA_GRID_2D, truth_alpha_2d, color="black", lw=2.2, label="truth"
)
axes[1, 1].plot(
    ALPHA_GRID_2D,
    likelihood_raw_alpha_2d,
    color=NORM_COLOR,
    lw=1.4,
    ls=":",
    label=r"CE $q_L$, before $Z_L$",
)
axes[1, 1].plot(
    ALPHA_GRID_2D,
    proposal_alpha_2d,
    color="0.55",
    ls=":",
    label=r"raw $q_P$",
)
axes[1, 1].plot(
    ALPHA_GRID_2D,
    hybrid_alpha_2d,
    color=JOINT_CE_COLOR,
    lw=1.9,
    label="CE posterior route",
)
axes[1, 1].plot(
    ALPHA_GRID_2D,
    likelihood_hybrid_alpha_2d,
    color=POSTHOC_COLOR,
    lw=1.7,
    ls="--",
    label=r"CE $q_L/Z_L$",
)
axes[1, 1].set(
    xlabel=r"$\alpha$",
    ylabel="posterior density",
    title="(d) Nuisance marginal",
)
for ax in axes.flat:
    ax.grid(alpha=0.22)
    if ax in axes[1]:
        ax.legend(fontsize=8)
export_exercise9_multiclass_figure(fig, "joint_three_class_posterior_closure")
plt.show()


## Held-out calibration of the joint posterior

For each fresh $(\theta_i,x_i)$, samples are drawn directly from $q_P(\mu,\alpha\mid x_i)$.  The corrected empirical distribution uses $d_S/d_P$ weights.  Marginal PIT and equal-tailed coverage are reported for both $\mu$ and $\alpha$.


In [ ]:
rng = np.random.default_rng(SEED + 1300)
theta_joint_calibration = sample_design(
    N_JOINT_CALIBRATION_CONTEXTS, rng
)
x_joint_calibration = simulate(theta_joint_calibration, rng)
theta_joint_draws = _draw_conditional(
    q_p_joint,
    x_joint_calibration,
    N_JOINT_CALIBRATION_SAMPLES,
    SEED + 1301,
)
joint_calibration_points = np.concatenate(
    [
        theta_joint_draws,
        np.repeat(
            x_joint_calibration[:, None, :],
            N_JOINT_CALIBRATION_SAMPLES,
            axis=1,
        ),
    ],
    axis=2,
)
joint_probabilities = predict_class_probabilities(
    joint_ce, joint_calibration_points
)
joint_ratios = class_probability_ratio(joint_probabilities, 0, 1)
joint_weights = normalized_probability_ratios(joint_ratios, axis=1)

joint_pits = {
    "raw joint q_P": {},
    JOINT_CE_LABEL: {},
}
for parameter_index, parameter_name in enumerate(("mu", "alpha")):
    below = (
        theta_joint_draws[..., parameter_index]
        <= theta_joint_calibration[:, parameter_index, None]
    )
    joint_pits["raw joint q_P"][parameter_name] = below.mean(axis=1)
    joint_pits[JOINT_CE_LABEL][parameter_name] = np.sum(
        joint_weights * below, axis=1
    )

JOINT_NOMINAL_COVERAGE = np.linspace(0.05, 0.95, 19)
joint_coverage = {}
joint_calibration_rows = []
for method, parameter_pits in joint_pits.items():
    joint_coverage[method] = {}
    for parameter_name, pit in parameter_pits.items():
        coverage = np.array(
            [
                np.mean(
                    (pit >= 0.5 * (1.0 - level))
                    & (pit <= 0.5 * (1.0 + level))
                )
                for level in JOINT_NOMINAL_COVERAGE
            ]
        )
        joint_coverage[method][parameter_name] = coverage
        ordered = np.sort(pit)
        uniform = (np.arange(len(ordered)) + 0.5) / len(ordered)
        joint_calibration_rows.append(
            {
                "method": method,
                "parameter": parameter_name,
                "PIT KS distance": float(
                    np.max(np.abs(ordered - uniform))
                ),
                "max coverage error": float(
                    np.max(
                        np.abs(coverage - JOINT_NOMINAL_COVERAGE)
                    )
                ),
            }
        )
display(pd.DataFrame(joint_calibration_rows).style.format(precision=4))

fig, axes = plt.subplots(2, 2, figsize=(11.0, 8.4), constrained_layout=True)
joint_colors = {"raw joint q_P": "0.55", JOINT_CE_LABEL: JOINT_CE_COLOR}
for column, parameter_name in enumerate(("mu", "alpha")):
    for method in joint_pits:
        pit = joint_pits[method][parameter_name]
        ordered = np.sort(pit)
        empirical = np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
        axes[0, column].plot(
            ordered,
            empirical,
            color=joint_colors[method],
            lw=1.8,
            label=method,
        )
        axes[1, column].plot(
            JOINT_NOMINAL_COVERAGE,
            joint_coverage[method][parameter_name],
            color=joint_colors[method],
            lw=1.8,
            label=method,
        )
    axes[0, column].plot([0, 1], [0, 1], "k--", lw=1)
    axes[1, column].plot([0, 1], [0, 1], "k--", lw=1)
    axes[0, column].set(
        xlabel=f"{parameter_name} PIT",
        ylabel="empirical CDF",
        title=f"({chr(97 + column)}) {parameter_name} PIT",
    )
    axes[1, column].set(
        xlabel="nominal coverage",
        ylabel="empirical coverage",
        title=f"({chr(99 + column)}) {parameter_name} coverage",
    )
for ax in axes.flat:
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.grid(alpha=0.22)
    ax.legend(fontsize=8)
export_exercise9_multiclass_figure(fig, "joint_three_class_calibration")
plt.show()


## Post-training normalization and bridge diagnostics

For the joint three-class construction the normalized bridge is

$$
\ell_B(\theta,x)=\log\rho(\theta)+\log q_L(x\mid\theta)
-\log q_P(\theta\mid x)+\log d_P-\log d_L
+\log Z_P(x)-\log Z_L(\theta).
$$

At the population solution this equals $\log m_\rho(x)$ and is constant in both $\mu$ and $\alpha$ at fixed $x$.  Fresh proposal banks estimate $Z_P$, $Z_L$, and the within-$x$ bridge variation.  These quantities are not training losses.


In [ ]:
def build_joint_bridge_bundle(
    n_groups, n_inner, n_mass_inner, q_p_joint, q_l, seed
):
    rng = np.random.default_rng(seed)
    theta_anchor = sample_tail_enriched_design(n_groups, rng)
    x_anchor = simulate(theta_anchor, rng)
    theta = _draw_conditional(
        q_p_joint, x_anchor, n_inner, seed + 1
    )
    x_repeat = np.repeat(x_anchor[:, None, :], n_inner, axis=1)
    points = np.concatenate([theta, x_repeat], axis=2).astype(np.float32)
    flat_theta = theta.reshape(-1, 2)
    flat_x = x_repeat.reshape(-1, 3)
    bridge_base = (
        design_logpdf(flat_theta)
        + _flow_log_prob(q_l, flat_x, context=flat_theta)
        - _flow_log_prob(q_p_joint, flat_theta, context=flat_x)
    ).reshape(n_groups, n_inner)
    x_zl = _draw_conditional(
        q_l, flat_theta, n_mass_inner, seed + 2
    )
    theta_zl = np.repeat(
        flat_theta[:, None, :], n_mass_inner, axis=1
    )
    zl_points = np.concatenate([theta_zl, x_zl], axis=2).reshape(
        n_groups, n_inner, n_mass_inner, 5
    )
    return {
        "points": points,
        "base": bridge_base,
        "zl_points": zl_points,
    }


joint_bridge_banks = [
    build_joint_bridge_bundle(
        N_BRIDGE_AUDIT_GROUPS,
        N_BRIDGE_AUDIT_INNER,
        N_BRIDGE_AUDIT_MASS_INNER,
        q_p_joint,
        q_l,
        SEED + 220_000 + 100 * bank,
    )
    for bank in range(N_BRIDGE_AUDIT_BANKS)
]
joint_bridge_rms = []
for bundle in joint_bridge_banks:
    log_probabilities = predict_class_log_probabilities(
        joint_ce, bundle["points"]
    )
    zl_probabilities = predict_class_probabilities(
        joint_ce, bundle["zl_points"]
    )
    log_zl = np.log(
        np.mean(class_probability_ratio(zl_probabilities, 0, 2), axis=2)
    )
    implied = (
        bundle["base"]
        + log_probabilities[..., 1]
        - log_probabilities[..., 2]
        - log_zl
    )
    joint_bridge_rms.append(np.std(implied, axis=1, ddof=1))
joint_bridge_rms = np.concatenate(joint_bridge_rms)
del joint_bridge_banks

rng = np.random.default_rng(SEED + 1190)
n_z_reference = N_DIAGNOSTIC_REFERENCE
theta_z_check = sample_design(
    32 if SMOKE_MODE else (80 if FAST_MODE else 160), rng
)
x_z_check = simulate(theta_z_check, rng)
joint_heldout_log_z = {
    r"$\log Z_P(x)$": joint_log_zp(
        joint_ce, x_z_check, n_z_reference, SEED + 1191
    ),
    r"$\log Z_L(\theta)$": joint_log_zl(
        joint_ce, theta_z_check, n_z_reference, SEED + 1192
    ),
}

theta_mode = THETA_GRID[np.argmax(posterior_truth_2d)]
theta_p_tail = _draw_conditional(
    q_p_joint,
    X_OBS[None, :],
    2_000 if SMOKE_MODE else (20_000 if FAST_MODE else 80_000),
    SEED + 1193,
)[0]
p_tail_points = np.column_stack(
    [theta_p_tail, np.repeat(X_OBS[None, :], len(theta_p_tail), axis=0)]
)
p_tail_probabilities = predict_class_probabilities(joint_ce, p_tail_points)
p_tail_ratios = class_probability_ratio(p_tail_probabilities, 0, 1)
p_tail_log_probabilities = np.log(
    np.maximum(p_tail_probabilities, np.finfo(np.float64).tiny)
)
p_tail_summary = importance_tail_summary(
    class_log_ratio(p_tail_log_probabilities, 0, 1)
)

x_l_tail = _draw_conditional(
    q_l, theta_mode[None, :], len(theta_p_tail), SEED + 1194
)[0]
l_tail_points = np.column_stack(
    [np.repeat(theta_mode[None, :], len(x_l_tail), axis=0), x_l_tail]
)
l_tail_probabilities = predict_class_probabilities(joint_ce, l_tail_points)
l_tail_log_probabilities = np.log(
    np.maximum(l_tail_probabilities, np.finfo(np.float64).tiny)
)
l_tail_summary = importance_tail_summary(
    class_log_ratio(l_tail_log_probabilities, 0, 2)
)

joint_diagnostic_rows = []
for name, values in joint_heldout_log_z.items():
    joint_diagnostic_rows.append(
        {
            "diagnostic": name,
            "median |value|": float(np.median(np.abs(values))),
            "q95 |value|": float(np.quantile(np.abs(values), 0.95)),
            "max |value|": float(np.max(np.abs(values))),
            "ESS fraction": np.nan,
            "Pareto k": np.nan,
            "max weight fraction": np.nan,
            "probability-floor fraction": np.nan,
        }
    )
joint_diagnostic_rows.extend(
    [
        {
            "diagnostic": "normalized bridge RMS",
            "median |value|": float(np.median(joint_bridge_rms)),
            "q95 |value|": float(np.quantile(joint_bridge_rms, 0.95)),
            "max |value|": float(np.max(joint_bridge_rms)),
            "ESS fraction": np.nan,
            "Pareto k": np.nan,
            "max weight fraction": np.nan,
            "probability-floor fraction": np.nan,
        },
        {
            "diagnostic": "posterior ratio tail",
            "median |value|": np.nan,
            "q95 |value|": np.nan,
            "max |value|": np.nan,
            "ESS fraction": p_tail_summary["ESS_fraction"],
            "Pareto k": p_tail_summary["pareto_k"],
            "max weight fraction": p_tail_summary["max_weight_fraction"],
            "probability-floor fraction": probability_floor_fraction(
                p_tail_probabilities, 1
            ),
        },
        {
            "diagnostic": "likelihood ratio tail",
            "median |value|": np.nan,
            "q95 |value|": np.nan,
            "max |value|": np.nan,
            "ESS fraction": l_tail_summary["ESS_fraction"],
            "Pareto k": l_tail_summary["pareto_k"],
            "max weight fraction": l_tail_summary["max_weight_fraction"],
            "probability-floor fraction": probability_floor_fraction(
                l_tail_probabilities, 2
            ),
        },
    ]
)
display(pd.DataFrame(joint_diagnostic_rows).style.format(precision=4))

MU_PATH = np.linspace(-3.5, 3.5, 181)
THETA_MU_PATH = np.column_stack([MU_PATH, np.full_like(MU_PATH, theta_mode[1])])
ALPHA_PATH = np.linspace(-3.0, 3.0, 161)
THETA_ALPHA_PATH = np.column_stack(
    [np.full_like(ALPHA_PATH, theta_mode[0]), ALPHA_PATH]
)

def joint_bridge_path(theta_path, seed):
    x = np.repeat(X_OBS[None, :], len(theta_path), axis=0)
    points = np.column_stack([theta_path, x])
    log_probabilities = predict_class_log_probabilities(joint_ce, points)
    raw = (
        design_logpdf(theta_path)
        + _flow_log_prob(q_l, x, context=theta_path)
        - _flow_log_prob(q_p_joint, theta_path, context=x)
        + log_probabilities[:, 1]
        - log_probabilities[:, 2]
    )
    normalized = (
        raw
        + log_zp_observed
        - joint_log_zl(
            joint_ce,
            theta_path,
            N_DIAGNOSTIC_REFERENCE,
            seed,
        )
    )
    return raw, normalized


raw_mu_bridge, normalized_mu_bridge = joint_bridge_path(
    THETA_MU_PATH, SEED + 1200
)
raw_alpha_bridge, normalized_alpha_bridge = joint_bridge_path(
    THETA_ALPHA_PATH, SEED + 1201
)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12.0, 8.6), constrained_layout=True)
axes[0, 0].plot(
    MU_PATH,
    raw_mu_bridge - LOG_EVIDENCE_TRUTH,
    color="0.55",
    ls=":",
    label="raw bridge",
)
axes[0, 0].plot(
    MU_PATH,
    normalized_mu_bridge - LOG_EVIDENCE_TRUTH,
    color=JOINT_CE_COLOR,
    lw=1.9,
    label="post-hoc normalized",
)
axes[0, 0].axhline(0, color="black", lw=1)
axes[0, 0].set(
    xlabel=r"$\mu$",
    ylabel="log-evidence residual",
    title="(a) Bridge along the POI",
)

axes[0, 1].plot(
    ALPHA_PATH,
    raw_alpha_bridge - LOG_EVIDENCE_TRUTH,
    color="0.55",
    ls=":",
    label="raw bridge",
)
axes[0, 1].plot(
    ALPHA_PATH,
    normalized_alpha_bridge - LOG_EVIDENCE_TRUTH,
    color=JOINT_CE_COLOR,
    lw=1.9,
    label="post-hoc normalized",
)
axes[0, 1].axhline(0, color="black", lw=1)
axes[0, 1].set(
    xlabel=r"$\alpha$",
    ylabel="log-evidence residual",
    title="(b) Bridge along the nuisance",
)

rng_plot = np.random.default_rng(SEED + 1202)
for index, (label, values) in enumerate(joint_heldout_log_z.items()):
    axes[1, 0].scatter(
        index + rng_plot.normal(0, 0.012, len(values)),
        values,
        s=12,
        alpha=0.4,
        color=JOINT_CE_COLOR,
    )
axes[1, 0].axhline(0, color="black", lw=1)
axes[1, 0].set(
    xticks=[0, 1],
    xticklabels=list(joint_heldout_log_z),
    ylabel="held-out log normalizer",
    title="(c) Conditional mass checks",
)

normalized_tail_ratios = p_tail_ratios / np.mean(p_tail_ratios)
ordered = np.sort(np.maximum(normalized_tail_ratios, np.finfo(float).tiny))
survival = 1.0 - np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
axes[1, 1].plot(ordered, survival, color=JOINT_CE_COLOR, lw=1.8)
axes[1, 1].set(
    xscale="log",
    yscale="log",
    xlabel=r"normalized $d_S/d_P$",
    ylabel="empirical survival",
    title="(d) Joint posterior-correction tail",
)
for ax in axes.flat:
    ax.grid(alpha=0.22)
    if ax in axes[0]:
        ax.legend(fontsize=8)
export_exercise9_multiclass_figure(
    fig, "joint_three_class_consistency_and_normalization"
)
plt.show()


## Systematic-prior and auxiliary-measurement update

Because $q_P$ and its correction retain the full $(\mu,\alpha)$ vector, the nuisance prior can be changed and an auxiliary likelihood added without retraining.  This is ordinary importance reweighting of the frozen corrected joint posterior.


In [ ]:
ALPHA_PRIOR_MEAN, ALPHA_PRIOR_SIGMA = 0.30, 0.45
A_OBSERVED, SIGMA_A = 0.10, 0.25
log_update_factor = (
    norm.logpdf(
        THETA_GRID[:, 1], loc=ALPHA_PRIOR_MEAN, scale=ALPHA_PRIOR_SIGMA
    )
    - design_alpha_logpdf(THETA_GRID[:, 1])
    + norm.logpdf(A_OBSERVED, loc=THETA_GRID[:, 1], scale=SIGMA_A)
)
learned_update, _ = normalize_log_surface(
    (
        np.log(np.maximum(posterior_joint_hybrid.ravel(), 1.0e-300))
        + log_update_factor
    ).reshape(MU_MESH.shape),
    MU_GRID_2D,
    ALPHA_GRID_2D,
)
truth_update, _ = normalize_log_surface(
    (
        log_likelihood(X_OBS, THETA_GRID)
        + design_mu_logpdf(THETA_GRID[:, 0])
        + norm.logpdf(
            THETA_GRID[:, 1],
            loc=ALPHA_PRIOR_MEAN,
            scale=ALPHA_PRIOR_SIGMA,
        )
        + norm.logpdf(
            A_OBSERVED, loc=THETA_GRID[:, 1], scale=SIGMA_A
        )
    ).reshape(MU_MESH.shape),
    MU_GRID_2D,
    ALPHA_GRID_2D,
)
learned_update_mu, learned_update_alpha = marginals(learned_update)
truth_update_mu, truth_update_alpha = marginals(truth_update)

fig, axes = plt.subplots(1, 2, figsize=(11.3, 4.3), constrained_layout=True)
axes[0].plot(MU_GRID_2D, truth_update_mu, color="black", lw=2.2, label="truth")
axes[0].plot(
    MU_GRID_2D,
    learned_update_mu,
    color=JOINT_CE_COLOR,
    lw=1.9,
    label="updated joint hNPE",
)
axes[0].set(
    xlabel=r"$\mu$", ylabel="posterior density", title="(a) Updated POI"
)
axes[1].plot(
    ALPHA_GRID_2D,
    truth_update_alpha,
    color="black",
    lw=2.2,
    label="truth",
)
axes[1].plot(
    ALPHA_GRID_2D,
    learned_update_alpha,
    color=JOINT_CE_COLOR,
    lw=1.9,
    label="updated joint hNPE",
)
axes[1].set(
    xlabel=r"$\alpha$",
    ylabel="posterior density",
    title="(b) Updated nuisance",
)
for ax in axes:
    ax.grid(alpha=0.22)
    ax.legend(fontsize=8)
export_exercise9_multiclass_figure(fig, "joint_systematic_update")
plt.show()


# Applications of the frozen joint dual model

The remaining cells keep the applications requested for the ML study: corrected likelihood generation, absolute evidence, posterior-predictive generation, and selection integrals.  Classifier corrections always use the direct softmax probability quotients.  Stable softmax normalization of generic log-density importance weights is used only when design and flow densities also enter an integral.


In [ ]:
N_APPLICATION = 256 if SMOKE_MODE else (2_048 if FAST_MODE else 4_096)
N_APPLICATION_Z = 32 if SMOKE_MODE else (64 if FAST_MODE else 128)
N_GENERATIVE = 512 if SMOKE_MODE else (5_000 if FAST_MODE else 20_000)


def corrected_likelihood_candidates(theta, n_candidates, seed):
    theta = np.atleast_2d(np.asarray(theta, dtype=np.float32))
    x = _draw_conditional(q_l, theta, n_candidates, seed)
    points = np.concatenate(
        [np.repeat(theta[:, None, :], n_candidates, axis=1), x], axis=2
    )
    probabilities = predict_class_probabilities(joint_ce, points)
    ratios = class_probability_ratio(probabilities, 0, 2)
    weights = normalized_probability_ratios(ratios, axis=1)
    return x, weights, np.log(np.mean(ratios, axis=1))


def importance_resample(values, weights, seed):
    values = np.asarray(values)
    weights = np.asarray(weights, dtype=float)
    rng = np.random.default_rng(seed)
    index = rng.choice(
        len(values), size=len(values), replace=True, p=weights / weights.sum()
    )
    return values[index]


## Corrected generation, evidence, and posterior prediction

Corrected likelihood generation uses $q_L$ candidates with $d_S/d_L$ resampling weights.  Evidence is evaluated from joint $q_P(\theta\mid x_o)$ proposals and the normalized corrected likelihood.  Posterior prediction is constructed once through the hNPE correction and once through the likelihood/evidence route.


In [ ]:
APP_THETA = np.array(
    [theta_mode, [-1.35, 0.75], [2.10, -0.60]], dtype=np.float32
)
rng = np.random.default_rng(SEED + 1500)
proposal_generations, corrected_generations, truth_generations = [], [], []
generative_rows = []
for index, theta in enumerate(APP_THETA):
    candidates, weights, log_z = corrected_likelihood_candidates(
        theta[None, :], N_GENERATIVE, SEED + 1501 + index
    )
    corrected = importance_resample(
        candidates[0], weights[0], SEED + 1510 + index
    )
    truth = simulate(
        np.repeat(theta[None, :], N_GENERATIVE, axis=0), rng
    )
    proposal_generations.append(candidates[0])
    corrected_generations.append(corrected)
    truth_generations.append(truth)
    generative_rows.append(
        {
            "theta": tuple(theta),
            "log Z_L": float(log_z[0]),
            "proposal mean W1": float(
                np.mean(
                    [
                        wasserstein_distance(
                            candidates[0, :, j], truth[:, j]
                        )
                        for j in range(3)
                    ]
                )
            ),
            "corrected mean W1": float(
                np.mean(
                    [
                        wasserstein_distance(
                            corrected[:, j], truth[:, j]
                        )
                        for j in range(3)
                    ]
                )
            ),
        }
    )
display(pd.DataFrame(generative_rows).style.format(precision=4))

theta_app = _draw_conditional(
    q_p_joint, X_OBS[None, :], N_APPLICATION, SEED + 1530
)[0].astype(np.float32)
x_obs_app = np.repeat(X_OBS[None, :], N_APPLICATION, axis=0).astype(
    np.float32
)
obs_points = np.column_stack([theta_app, x_obs_app])
obs_probabilities = predict_class_probabilities(joint_ce, obs_points)
obs_log_probabilities = np.log(
    np.maximum(obs_probabilities, np.finfo(np.float64).tiny)
)
rp_obs = class_probability_ratio(obs_probabilities, 0, 1)
log_rl_obs = class_log_ratio(obs_log_probabilities, 0, 2)
log_qp_app = _flow_log_prob(q_p_joint, theta_app, context=x_obs_app)
log_ql_app = _flow_log_prob(q_l, x_obs_app, context=theta_app)
zl_app = joint_zl(
    joint_ce, theta_app, N_APPLICATION_Z, SEED + 11_530
)
log_likelihood_app = log_ql_app + log_rl_obs - np.log(zl_app)
log_evidence_weight = (
    design_logpdf(theta_app)
    + log_likelihood_app
    - log_qp_app
)
learned_log_evidence = float(
    logsumexp(log_evidence_weight) - np.log(N_APPLICATION)
)
evidence_tail = importance_tail_summary(log_evidence_weight)
print(
    f"absolute log evidence: learned={learned_log_evidence:.5f}, "
    f"truth={LOG_EVIDENCE_TRUTH:.5f}; "
    f"ESS fraction={evidence_tail['ESS_fraction']:.4f}, "
    f"Pareto k={evidence_tail['pareto_k']:.3f}"
)

EVIDENCE_SIZES = np.unique(np.geomspace(32, N_APPLICATION, 8).astype(int))
evidence_trace = np.array(
    [
        logsumexp(log_evidence_weight[:n]) - np.log(n)
        for n in EVIDENCE_SIZES
    ]
)

x_rep = _draw_conditional(q_l, theta_app, 1, SEED + 1540)[:, 0, :]
rep_points = np.column_stack([theta_app, x_rep])
rep_probabilities = predict_class_probabilities(joint_ce, rep_points)
rep_log_probabilities = np.log(
    np.maximum(rep_probabilities, np.finfo(np.float64).tiny)
)
rl_rep = class_probability_ratio(rep_probabilities, 0, 2)
log_rl_rep = class_log_ratio(rep_log_probabilities, 0, 2)

predictive_hnpe_weights = normalized_probability_ratios(
    rp_obs * (rl_rep / zl_app), axis=0
)
predictive_hnde_weights = normalized_log_weights(
    log_evidence_weight + log_rl_rep - np.log(zl_app)
)
predictive_hnpe = importance_resample(
    x_rep, predictive_hnpe_weights, SEED + 1541
)
predictive_hnde = importance_resample(
    x_rep, predictive_hnde_weights, SEED + 1542
)
rng_truth = np.random.default_rng(SEED + 1543)
truth_index = rng_truth.choice(
    len(THETA_GRID),
    size=N_APPLICATION,
    replace=True,
    p=posterior_truth_2d.ravel() / posterior_truth_2d.sum(),
)
predictive_truth = simulate(THETA_GRID[truth_index], rng_truth)

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.3), constrained_layout=True)
for index, (proposal, corrected, truth) in enumerate(
    zip(proposal_generations, corrected_generations, truth_generations)
):
    axes[0].hist(
        truth[:, 1], bins=55, density=True, histtype="step", lw=1.6
    )
    axes[0].hist(
        proposal[:, 1],
        bins=55,
        density=True,
        histtype="step",
        lw=1.0,
        ls=":",
    )
    axes[0].hist(
        corrected[:, 1],
        bins=55,
        density=True,
        histtype="step",
        lw=1.5,
        ls="--",
    )
axes[0].set(
    xlabel=r"$x_2$",
    ylabel="density",
    title="(a) Corrected likelihood generation",
)

axes[1].plot(
    EVIDENCE_SIZES,
    evidence_trace,
    marker="o",
    color=JOINT_CE_COLOR,
    label="learned",
)
axes[1].axhline(
    LOG_EVIDENCE_TRUTH, color="black", ls="--", label="truth"
)
axes[1].set(
    xscale="log",
    xlabel="outer q_P samples",
    ylabel="log evidence",
    title="(b) Absolute evidence",
)
axes[1].legend(fontsize=8)

axes[2].hist(
    predictive_truth[:, 1],
    bins=50,
    density=True,
    histtype="step",
    color="black",
    lw=2.0,
    label="truth",
)
axes[2].hist(
    predictive_hnpe[:, 1],
    bins=50,
    density=True,
    histtype="step",
    color=JOINT_CE_COLOR,
    lw=1.7,
    label="hNPE route",
)
axes[2].hist(
    predictive_hnde[:, 1],
    bins=50,
    density=True,
    histtype="step",
    color=POSTHOC_COLOR,
    lw=1.7,
    ls="--",
    label="hNDE route",
)
axes[2].set(
    xlabel=r"$x_2^{\rm rep}$",
    ylabel="density",
    title="(c) Posterior predictive",
)
axes[2].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.22)
export_exercise9_multiclass_figure(
    fig, "joint_generation_evidence_predictive"
)
plt.show()


## Selection integrals without simulator calls

For $T(x)=x_2-0.6x_3$ and selection $T>0.5$, draws from $q_L(x\mid\theta)$ are averaged with the direct $d_S/d_L$ probability ratio.  As before, the model must represent the pre-selection distribution and retain the selection statistic.


In [ ]:
N_SELECTION_GRID = 13 if SMOKE_MODE else (25 if FAST_MODE else 41)
N_SELECTION_DRAWS = 128 if SMOKE_MODE else (256 if FAST_MODE else 512)
MU_SELECTION = np.linspace(-3.2, 3.2, N_SELECTION_GRID)
ALPHA_SELECTION = np.linspace(-2.4, 2.4, N_SELECTION_GRID)
MU_SEL_MESH, ALPHA_SEL_MESH = np.meshgrid(
    MU_SELECTION, ALPHA_SELECTION, indexing="ij"
)
THETA_SELECTION = np.column_stack(
    [MU_SEL_MESH.ravel(), ALPHA_SEL_MESH.ravel()]
).astype(np.float32)


def learned_selection_efficiency(theta, n_draws, seed):
    theta = np.atleast_2d(np.asarray(theta, dtype=np.float32))
    x = _draw_conditional(q_l, theta, n_draws, seed)
    points = np.concatenate(
        [np.repeat(theta[:, None, :], n_draws, axis=1), x], axis=2
    )
    probabilities = predict_class_probabilities(joint_ce, points)
    ratios = class_probability_ratio(probabilities, 0, 2)
    selected = (x[..., 1] - 0.6 * x[..., 2]) > 0.5
    return np.sum(ratios * selected, axis=1) / np.sum(ratios, axis=1)


beta_learned = learned_selection_efficiency(
    THETA_SELECTION, N_SELECTION_DRAWS, SEED + 201_560
).reshape(MU_SEL_MESH.shape)
mean_t = (
    0.72 * MU_SEL_MESH**2
    - 0.48 * np.cos(MU_SEL_MESH)
    - 0.58 * ALPHA_SEL_MESH
)
sigma_t = np.sqrt(
    SIMULATOR_SIGMA[1] ** 2 + 0.6**2 * SIMULATOR_SIGMA[2] ** 2
)
beta_truth = 1.0 - norm.cdf((0.5 - mean_t) / sigma_t)

fig, axes = plt.subplots(1, 3, figsize=(13.7, 4.2), constrained_layout=True)
extent = [
    ALPHA_SELECTION[0],
    ALPHA_SELECTION[-1],
    MU_SELECTION[0],
    MU_SELECTION[-1],
]
im0 = axes[0].imshow(
    beta_truth,
    origin="lower",
    aspect="auto",
    extent=extent,
    vmin=0,
    vmax=1,
    cmap="viridis",
)
axes[1].imshow(
    beta_learned,
    origin="lower",
    aspect="auto",
    extent=extent,
    vmin=0,
    vmax=1,
    cmap="viridis",
)
residual = beta_learned - beta_truth
bound = max(0.02, float(np.max(np.abs(residual))))
im2 = axes[2].imshow(
    residual,
    origin="lower",
    aspect="auto",
    extent=extent,
    vmin=-bound,
    vmax=bound,
    cmap="coolwarm",
)
axes[0].set_title("(a) Analytic validation truth")
axes[1].set_title("(b) Corrected joint hNDE")
axes[2].set_title("(c) Learned minus truth")
for ax in axes:
    ax.set(xlabel=r"$\alpha$", ylabel=r"$\mu$")
fig.colorbar(im0, ax=axes[:2], label=r"selection efficiency $\beta$")
fig.colorbar(im2, ax=axes[2], label="efficiency residual")
export_exercise9_multiclass_figure(fig, "joint_selection_integral")
plt.show()


## Conclusions and run checklist

This refactor returns the demonstration to the minimal dual construction:

1. Part I marginalizes $\alpha$ implicitly and retains the scalar $q_P^m(\mu\mid x)$ proposal;
2. Part II trains the original joint posterior flow $q_P(\mu,\alpha\mid x)$ and likelihood flow $q_L(x\mid\mu,\alpha)$, with the original Exercise-9 architecture and maximum-likelihood training;
3. both corrections use ten independently initialized plain three-output MLPs trained with equal-prior multiclass CE only, and inference averages their positive softmax ratios arithmetically;
4. $\mathcal L_a-\log2$, dropout, weight decay, layer normalization, residual heads, output bounds, normalization losses, and bridge losses are absent;
5. residuals are obtained from float64 softmax probability quotients, without explicit exponentiation of logit differences;
6. normalization and bridge relations remain post-training corrections/checks, while simulator closure and SBC remain the external validation.

A full paper run should repeat the complete flow and classifier training over independent seeds and inspect probability floors, ESS/Pareto-$k$, conditional masses, bridge variation, and calibration before using any downstream application.
